# Structured constant-volatility path generator


## Install libraries

Run the next cell to install the tools we need:

- **pandas** works with tables of data.
- **matplotlib** creates charts and plots.
- **numpy** works with arrays of numbers and calculations.
- **modelx** builds account models whose calculated totals update when inputs change.
- **pandas_market_calendars** supplies exchange trading dates, including holidays.
- **pyarrow** saves and reads Parquet files.


In [ ]:
%pip install pandas matplotlib numpy modelx pandas_market_calendars pyarrow


## Import libraries

Imports make the installed tools available in this notebook. The short names `pd`, `plt`, and `np` keep the code easy to write.

`json` reads JSON files, and `Path` handles file locations. Both are included with Python.

`mcal` supplies the exchange calendar used for daily stock-price paths.


In [ ]:
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import pandas_market_calendars as mcal


## Simulation setup

Select a company with `TICKER` and an annual record with `FINANCIAL_YEAR`. `N_PATHS` controls the number of scenarios. The simulation uses NYSE trading dates and calendar-day weights for annual accounting shocks. `EXPORT_ROOT` selects the output directory.

The accounts are fiscal-year inputs, not necessarily calendar-year accounts. Existing forecasts retain the simplified fixed-cost, net-profit-as-cash-change assumptions. Historical share counts follow each company's source basis; where required, the notebook explicitly adjusts them to the split-adjusted price basis without changing source data.

This is retrospective scenario calibration, not a point-in-time backtest. Financial statements, credit ratings, and hedge schedules can use information published after the simulation start. `input_snapshot_audit` exposes financial and hedge snapshot dates; those dates do not establish publication availability.

Run `prepare_joint_price_innovations.ipynb` first for the desired simulation year(s).
Its path count, master seed, history settings and innovation root must match this notebook.
The default example is AAPL because NOG is not in the project's fixed 30-company portfolio.
Only complete calendar years are supported by the shared annual dataset.


In [ ]:
SIMULATION_START_DATE = pd.Timestamp("2026-01-01")
SIMULATION_DURATION_DAYS = 365
SIMULATION_END_DATE = SIMULATION_START_DATE + pd.Timedelta(days=SIMULATION_DURATION_DAYS)
TICKER = "AAPL"  # First member of the configured 30-company universe.
N_PATHS = 10_000
EXPORT_ROOT = Path("export")
NONPOSITIVE_ASSET_VALUE_POLICY = "limited_liability_zero"
FINANCIAL_YEAR = 2025  # Fiscal year of the income statement and balance sheet.

market_calendar = mcal.get_calendar("NYSE")
trading_dates = market_calendar.valid_days(
    start_date=SIMULATION_START_DATE,
    end_date=SIMULATION_END_DATE - pd.Timedelta(days=1),
).tz_localize(None)
TRADING_DAYS = len(trading_dates)
if TRADING_DAYS == 0:
    raise ValueError("The simulation period must contain at least one trading day.")

if isinstance(N_PATHS, bool) or not isinstance(N_PATHS, int) or N_PATHS < 1:
    raise ValueError("N_PATHS must be a positive integer.")

# Shared price innovations are prepared once, separately from company simulations.
# The simulation year (not FINANCIAL_YEAR) selects the pre-year historical window.
JOINT_INNOVATION_ROOT = Path("data/joint_price_innovations")
PRICE_MASTER_SEED = 2026
CORRELATION_LOOKBACK_YEARS = 5
CORRELATION_SHRINKAGE = 0.05
CORRELATION_MIN_OBSERVATIONS = 500
from runpy import run_path
joint_innovation_functions = run_path(str(Path("accounting-scripts") / "joint_innovations.py"))
SIMULATION_YEAR = int(SIMULATION_START_DATE.year)
if (
    SIMULATION_START_DATE != pd.Timestamp(year=SIMULATION_YEAR, month=1, day=1)
    or SIMULATION_END_DATE != pd.Timestamp(year=SIMULATION_YEAR + 1, month=1, day=1)
):
    raise ValueError("Joint innovations require a full calendar year; use 366 days for a leap year.")
if TICKER not in joint_innovation_functions["COMPANIES"]:
    raise ValueError("Select TICKER from joint_innovation_functions['COMPANIES']; NOG is outside this 30-company universe.")
joint_innovation_config = joint_innovation_functions["annual_config"](
    SIMULATION_YEAR, n_simulations=N_PATHS, master_seed=PRICE_MASTER_SEED,
    lookback_years=CORRELATION_LOOKBACK_YEARS, shrinkage=CORRELATION_SHRINKAGE,
    min_observations=CORRELATION_MIN_OBSERVATIONS,
)


### Predefined shock events

Each row defines an event's date, publisher, type, and change. Positive changes mean an increase; negative changes mean a decrease. Converting the dates lets us compare them with the simulation dates.

In [ ]:
shock_events_df = pd.DataFrame(
    [
        ("2026-02-28", "ZHAW Daily", "oil_price_shock", 0.48),
        ("2026-06-08", "ZHAW Daily", "oil_price_shock", -0.29),
        ("2026-09-17", "ZHAW Daily", "interest_rate_shock", 0.07),
    ],
    columns=["date", "publisher", "shock_type", "price_change"],
)

shock_events_df["date"] = pd.to_datetime(shock_events_df["date"])


## Load relevant data

We load the simulation inputs from the `data` folder beside this notebook. Run the notebook from its project folder so these file paths work. The small helper below reads JSON files.

In [ ]:
DATA_DIR = Path("data")

if not DATA_DIR.is_dir():
    raise FileNotFoundError("Run this notebook from the project folder containing 'data'.")


def load_json(relative_path):
    with (DATA_DIR / relative_path).open(encoding="utf-8") as file:
        return json.load(file)

# This loads only; missing/incompatible annual data is never generated silently.
company_price_innovations, joint_innovation_metadata = (
    joint_innovation_functions["load_company_innovations"](
        JOINT_INNOVATION_ROOT, joint_innovation_config, TICKER,
        trading_dates, pd.RangeIndex(N_PATHS),
    )
)
print(f"Shared annual price innovations: {joint_innovation_metadata['manifest_sha256']}")


### Financial data

`FINANCIAL_YEAR` selects `data/accounting/financials_<year>.json`. We filter that
file by `TICKER` and verify that the record's fiscal year matches the selection.
The matching `financials` dictionary keeps the income statement and balance sheet
from the same year together, including the cash reserves used in the simulation.
The file is loaded once and reused for all path models.


In [ ]:
if (
    isinstance(FINANCIAL_YEAR, bool)
    or not isinstance(FINANCIAL_YEAR, (int, np.integer))
    or not 1000 <= FINANCIAL_YEAR <= 9999
):
    raise ValueError("FINANCIAL_YEAR must be a four-digit integer, such as 2025.")

financials_data = [
    record
    for record in load_json(f"accounting/financials_{FINANCIAL_YEAR}.json")
    if record["ticker"] == TICKER
]
if len(financials_data) != 1:
    raise ValueError(
        f"Expected one financial record for {TICKER} in {FINANCIAL_YEAR}, "
        f"found {len(financials_data)}."
    )

financials = financials_data[0]
if financials.get("period") != FINANCIAL_YEAR:
    raise ValueError(f"The selected financial record does not report fiscal year {FINANCIAL_YEAR}.")

# A selected fiscal record is a scenario snapshot, not a point-in-time backtest.
financial_snapshot_date = pd.Timestamp(financials["balance_sheet"]["as_of_date"])
financial_snapshot_after_simulation_start = financial_snapshot_date >= SIMULATION_START_DATE
print(f"Accounting snapshot: {financial_snapshot_date:%Y-%m-%d}; simulation starts: {SIMULATION_START_DATE:%Y-%m-%d}")
if financial_snapshot_after_simulation_start:
    print("Snapshot scenario: these accounts were not available at the simulation start. This is not a historical backtest.")


### Stock prices and daily returns

`TICKER` selects the stock-price file. We keep the date and closing price, sort by date, and calculate two types of daily return:

- `daily_simple_return = close / previous_close - 1`: the ordinary percentage price change, used by the existing shock-beta calculations.
- `daily_log_return = ln(close / previous_close)`: the log return, used for daily volatility and available for later daily price-path simulation.

For example, a rise from $100 to $110 gives a simple return of `0.10` and a log return of approximately `0.09531`. The first return is missing because there is no earlier price.

The betas continue to predict simple stock returns. Before using a predicted shock in a log-return simulation, convert it with `np.log1p(predicted_simple_return)`. This requires a predicted simple return greater than `-1`. Convert a log return back to an ordinary percentage change with `np.expm1(log_return)`.

The supplied prices are unadjusted, so these returns do not correct for stock splits or dividends.


In [ ]:
stock_prices_data = load_json(f"prices/{TICKER}.json")
stock_prices_df = pd.DataFrame(stock_prices_data["data"])[["date", "close"]].copy()
stock_prices_df["date"] = pd.to_datetime(stock_prices_df["date"])
stock_prices_df = stock_prices_df.sort_values("date").reset_index(drop=True)

if not np.isfinite(stock_prices_df["close"]).all() or (stock_prices_df["close"] <= 0).any():
    raise ValueError("Stock log returns require finite, positive closing prices.")

stock_prices_df["daily_simple_return"] = stock_prices_df["close"].pct_change(fill_method=None)
stock_prices_df["daily_log_return"] = np.log(
    stock_prices_df["close"] / stock_prices_df["close"].shift(1)
)


### S&P default probabilities

This dictionary maps each S&P credit rating to its historical one-year default probability. Values are decimals: `0.0183` means `1.83%`.

In [ ]:
sp_default_probabilities = load_json("reference/sp_default_probabilities.json")[
    "default_probability_by_rating"
]


### Company rating and default probability

`TICKER` selects the company-information file. We read its credit rating and look up the matching value in `sp_default_probabilities`, storing it in `default_probability`.

In [ ]:
company_info = load_json(f"company-information/{TICKER}.json")
credit_rating = company_info["credit_rating"]["rating"]
default_probability = sp_default_probabilities[credit_rating]


### Crude oil hedge coverage

We load the crude oil hedge file for `TICKER` and keep only the coverage percentage, period start, and period end. The dates are converted and sorted for the chart.

In [ ]:
oil_hedges_df = pd.DataFrame(
    load_json(f"hedge-information/{TICKER.lower()}_crud_oil_hedge_coverage.json")
)[[
    "approximate_oil_hedge_coverage_pct",
    "hedge_coverage_period_start",
    "hedge_coverage_period_end",
]].copy()

for column in ("hedge_coverage_period_start", "hedge_coverage_period_end"):
    oil_hedges_df[column] = pd.to_datetime(oil_hedges_df[column])

oil_hedges_df = oil_hedges_df.sort_values("hedge_coverage_period_start").reset_index(drop=True)


Each horizontal line shows the coverage over its start-to-end period. The y-axis shows coverage in percent, and the x-axis shows time.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.hlines(
    y=oil_hedges_df["approximate_oil_hedge_coverage_pct"],
    xmin=oil_hedges_df["hedge_coverage_period_start"],
    xmax=oil_hedges_df["hedge_coverage_period_end"],
    color="tab:blue",
    linewidth=3,
)
ax.set_title(f"{TICKER}: Crude oil hedge coverage")
ax.set_xlabel("Time")
ax.set_ylabel("Coverage (%)")
ax.yaxis.set_major_formatter("{x:.0f}%")
ax.margins(x=0.01, y=0.08)
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.show()


### Interest rate hedge coverage

We load the interest rate hedge file for `TICKER` and keep only the coverage percentage, period start, and period end. The dates are converted and sorted for the chart.

In [ ]:
interest_rate_hedges_df = pd.DataFrame(
    load_json(f"hedge-information/{TICKER.lower()}_interest_rate_hedge_coverage.json")
)[[
    "approximate_interest_rate_hedge_coverage_pct",
    "hedge_coverage_period_start",
    "hedge_coverage_period_end",
]].copy()

for column in ("hedge_coverage_period_start", "hedge_coverage_period_end"):
    interest_rate_hedges_df[column] = pd.to_datetime(interest_rate_hedges_df[column])

interest_rate_hedges_df = interest_rate_hedges_df.sort_values("hedge_coverage_period_start").reset_index(drop=True)

input_snapshot_audit = pd.DataFrame([
    {"input": "financial_statements", "snapshot_date": financial_snapshot_date},
    *[{"input": "oil_hedges", "snapshot_date": pd.Timestamp(row["as_of_date"])}
      for row in load_json(f"hedge-information/{TICKER.lower()}_crud_oil_hedge_coverage.json")],
    *[{"input": "interest_hedges", "snapshot_date": pd.Timestamp(row["as_of_date"])}
      for row in load_json(f"hedge-information/{TICKER.lower()}_interest_rate_hedge_coverage.json")],
]).drop_duplicates().reset_index(drop=True)
input_snapshot_audit["snapshot_after_simulation_start"] = input_snapshot_audit["snapshot_date"] >= SIMULATION_START_DATE
# Snapshot dates are not filing/publication dates. The data is retrospective and
# is never represented as an ex-ante information set, even when all dates precede start.
input_snapshot_audit


Each horizontal line shows the coverage over its start-to-end period. The y-axis shows coverage in percent, and the x-axis shows time.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.hlines(
    y=interest_rate_hedges_df["approximate_interest_rate_hedge_coverage_pct"],
    xmin=interest_rate_hedges_df["hedge_coverage_period_start"],
    xmax=interest_rate_hedges_df["hedge_coverage_period_end"],
    color="tab:blue",
    linewidth=3,
)
ax.set_title(f"{TICKER}: Source interest-rate hedge ratios (protective or gross mixed-direction)")
ax.set_xlabel("Time")
ax.set_ylabel("Source notional ratio (%)")
ax.yaxis.set_major_formatter("{x:.0f}%")
ax.margins(x=0.01, y=0.08)
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.show()


### WTI spot prices

We read `prices/WTI.csv` into a separate DataFrame named `wti_prices_df`. The price column is renamed to `spot_price`; `observation_date` keeps its original name.

In [ ]:
wti_prices_df = pd.read_csv(DATA_DIR / "prices" / "WTI.csv").rename(
    columns={"DCOILWTICO": "spot_price"}
)


## Operational income statement

We call `generate_operational_income_statement(FINANCIAL_YEAR, financials=financials)`
to create a modelx object from the selected year's loaded financial record.
The method checks the year and records it as `model.financial_year`.
Calling `generate_operational_income_statement(year=2024)` directly also works:
it reads NOG's record from `financials_2024.json` itself.
Expenses stay fixed when revenue changes, while taxes and net profit recalculate.
Income tax is zero when profit before tax is zero or negative. Positive profit
before tax uses the original tax percentage, so losses receive no tax benefit.
Use `operational_statement` to access the accounts.


In [ ]:
from runpy import run_path

accounting_script = Path("accounting-scripts") / f"{TICKER}.py"
accounting_functions = run_path(str(accounting_script))
notebook_support = run_path(str(Path("accounting-scripts") / "notebook_support.py"))
generate_operational_income_statement = accounting_functions["generate_operational_income_statement"]
valuation_settings = notebook_support["prepare_valuation_settings"](financials, company_info)
if "operational_income_statement_model" in globals():
    operational_income_statement_model.close()
operational_income_statement_model = generate_operational_income_statement(
    FINANCIAL_YEAR, financials=financials, eps_settings=valuation_settings
)
price_valuation_settings = notebook_support["align_price_share_basis"](
    accounting_functions, operational_income_statement_model, financials, valuation_settings,
    split_adjusted=stock_prices_data.get("adjusted_for_splits", False),
)
if price_valuation_settings != valuation_settings:
    operational_income_statement_model.close()
    valuation_settings = price_valuation_settings
    operational_income_statement_model = generate_operational_income_statement(
        FINANCIAL_YEAR, financials=financials, eps_settings=valuation_settings
    )
operational_statement = operational_income_statement_model.IncomeStatement


### Revenue, net profit, and cash reserves

Each revenue level is a separate scenario for one reporting period. We treat negative net profit as cash burn. **Break-even** means net profit is zero. The **model default point** is where the loss uses up all cash; this is the minimum revenue boundary for avoiding cash exhaustion in that period.

At default, we assume the company stops operating. The line then stays flat at a loss equal to the starting cash reserves. This display rule does not change the income-statement model.

This is a simplified cash estimate because net profit includes noncash expenses. Income tax is zero on losses, so each dollar of pretax loss reduces cash by one dollar. Above zero pretax profit, the original tax percentage applies. The chart and default threshold use this rule.


In [ ]:
get_cash_amount = accounting_functions["get_cash_amount"]
cash_amount = float(get_cash_amount(financials["balance_sheet"]))
current_revenue = float(operational_statement.revenue())
current_net_profit = float(operational_statement.net_income())

# Solve the live company's income equations, including exceptional-tax offsets.
current_profit_before_tax = float(operational_statement.profit_before_tax())
income_tax_rate = float(operational_statement.tax_rate_pct()) / 100
reference_revenue = float(operational_statement.initial_revenue)
if not np.isfinite([cash_amount, current_revenue, current_net_profit, reference_revenue]).all() or cash_amount < 0:
    raise ValueError("Revenue thresholds require finite inputs and nonnegative opening cash.")
solve_revenue_for_profit = notebook_support["solve_revenue_for_profit"]
break_even_revenue = solve_revenue_for_profit(operational_income_statement_model, 0.0)
minimum_revenue = solve_revenue_for_profit(operational_income_statement_model, -cash_amount)

try:
    # Focus on the thresholds and the model's current revenue.
    revenue_span = max(
        abs(current_revenue - minimum_revenue),
        abs(break_even_revenue - minimum_revenue),
        reference_revenue * 0.02,
    )
    revenue_min = max(0.0, min(minimum_revenue, current_revenue) - 0.35 * revenue_span)
    revenue_max = max(break_even_revenue, current_revenue) + 0.25 * revenue_span
    revenues = np.unique(np.r_[
        np.linspace(revenue_min, revenue_max, 301),
        break_even_revenue, minimum_revenue, current_revenue,
    ])
    net_profits = []
    for revenue in revenues:
        operational_statement.revenue.value = float(revenue)
        net_profits.append(float(operational_statement.net_income()))
    net_profits = np.asarray(net_profits)
finally:
    # Leave the model at the same revenue as before drawing the chart.
    operational_statement.revenue.value = current_revenue


In [ ]:
million = 1_000_000
# After default, keep the displayed loss equal to the cash reserves used up.
plotted_net_profits = np.maximum(net_profits, -cash_amount)
plotted_current_net_profit = max(current_net_profit, -cash_amount)
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(revenues / million, plotted_net_profits / million, color="#224e80", lw=2.5, label="Net profit")
ax.axvspan(revenue_min / million, minimum_revenue / million,
           color="#c44141", alpha=0.10, label="Cash exhausted")
ax.fill_between(
    revenues / million, plotted_net_profits / million, 0,
    where=(revenues >= minimum_revenue) & (revenues <= break_even_revenue),
    color="#e8b04b", alpha=0.50, label="Loss covered by cash",
)
ax.axhline(0, color="#64748b", lw=1)
ax.hlines(-cash_amount / million, minimum_revenue / million, revenue_max / million,
          color="#bc862a", ls="--", lw=1.3)
ax.axvline(break_even_revenue / million, color="#23856d", ls=":", lw=1.4)
ax.axvline(minimum_revenue / million, color="#c44141", ls=":", lw=1.4)

ax.scatter(break_even_revenue / million, 0, color="#23856d", s=55, zorder=4)
ax.annotate(
    f"Break-even revenue\n${break_even_revenue / million:,.2f}m\nNet profit: $0",
    xy=(break_even_revenue / million, 0), xytext=(-8, 42),
    textcoords="offset points", ha="center", color="#16644f",
    bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.95, "pad": 3},
    arrowprops={"arrowstyle": "->", "color": "#23856d"},
)
ax.scatter(minimum_revenue / million, -cash_amount / million, color="#c44141", s=55, zorder=4)
ax.annotate(
    f"Minimum revenue / model default\n${minimum_revenue / million:,.2f}m\nCash remaining: $0",
    xy=(minimum_revenue / million, -cash_amount / million), xytext=(16, -40),
    textcoords="offset points", ha="left", color="#a32f2f",
    bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.95, "pad": 3},
    arrowprops={"arrowstyle": "->", "color": "#c44141"},
)
ax.scatter(current_revenue / million, plotted_current_net_profit / million, color="#224e80", s=45, zorder=4)
ax.annotate(
    f"Current revenue: ${current_revenue / million:,.2f}m",
    xy=(current_revenue / million, plotted_current_net_profit / million),
    xytext=(-8, 15), textcoords="offset points", ha="right", color="#224e80",
)
ax.annotate(
    "", xy=(0.96, 0), xytext=(0.96, -cash_amount / million),
    xycoords=ax.get_yaxis_transform(),
    arrowprops={"arrowstyle": "<->", "color": "#9b6c1c"},
)
ax.text(
    0.94, -cash_amount / (2 * million), f"Cash reserves\n${cash_amount / million:,.2f}m",
    transform=ax.get_yaxis_transform(), ha="right", va="center", color="#9b6c1c",
)
ax.set_title(f"{TICKER}: revenue, net profit, and cash reserves — one reporting period", pad=16)
ax.set_xlabel("Revenue (USD millions)")
ax.set_ylabel("Net profit (USD millions)")
ax.set_xlim(revenue_min / million, revenue_max / million)
ax.margins(y=0.25)
ax.ticklabel_format(axis="both", style="plain", useOffset=False)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(alpha=0.18)
ax.set_axisbelow(True)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=3, frameon=False)
fig.tight_layout()
plt.show()


### Calibrated revenue distribution

Positive observed default rates calibrate a normal revenue distribution around current revenue. A zero observed default rate has no finite positive-variance normal solution; the notebook uses its constant-revenue boundary case and records `calibration_method`. This does not claim those companies are risk-free: market prices still use historical volatility.

The normal model permits negative revenue in extreme scenarios and is a modeling assumption, especially for financial institutions. The target probability concerns end-of-period cash exhaustion under the profit cash proxy, not default at every earlier point in the year.

If the reported-revenue scenario already exhausts cash (for example, 3M fiscal 2023 under the profit cash proxy), a mean-centered normal distribution cannot match a rating probability below 50%. The notebook retains constant reported revenue, records the actual model default probability, and explicitly marks `rating_calibration_feasible=False`. It does not claim to have matched the rating or remove exceptional reported charges.


In [ ]:
revenue_mean = float(current_revenue)
target_default_probability = float(default_probability)
revenue_distribution = notebook_support["RevenueDistribution"](
    revenue_mean, float(minimum_revenue), target_default_probability
)
revenue_std = revenue_distribution.stdev
calibration_method = revenue_distribution.method
calibrated_default_probability = revenue_distribution.cdf(minimum_revenue)
cash_burning_probability = revenue_distribution.cdf(break_even_revenue) - calibrated_default_probability
profitable_probability = 1 - revenue_distribution.cdf(break_even_revenue)
planned_path_count = N_PATHS
expected_default_paths = planned_path_count * calibrated_default_probability
plot_span = max(4.5 * revenue_std, revenue_mean * 0.02)
distribution_revenues = np.unique(np.r_[
    np.linspace(revenue_mean - plot_span, revenue_mean + plot_span, 1601),
    minimum_revenue, break_even_revenue,
])
revenue_density = np.array([revenue_distribution.pdf(value) for value in distribution_revenues])
print(f"Revenue calibration: {calibration_method}")

revenue_calibration_audit = {
    "method": calibration_method,
    "rating_probability": target_default_probability,
    "model_probability": calibrated_default_probability,
    "rating_calibration_feasible": revenue_distribution.calibration_feasible,
}
if not revenue_distribution.calibration_feasible:
    print("Rating calibration is infeasible: reported profit already exhausts cash. Retaining the reported-revenue stress case; the rating probability is not matched.")
revenue_calibration_audit


In [ ]:
million = 1_000_000
fig, ax = plt.subplots(figsize=(12, 6))
x = distribution_revenues / million
density = revenue_density * million  # Density per USD million, matching the x-axis.

ax.fill_between(
    x, 0, density, where=distribution_revenues <= minimum_revenue,
    color="#d9534f", label=f"Default: {calibrated_default_probability:.2%}",
)
ax.fill_between(
    x, 0, density,
    where=(distribution_revenues >= minimum_revenue) & (distribution_revenues <= break_even_revenue),
    color="#f3ce49", label=f"Cash burning: {cash_burning_probability:.2%}",
)
ax.fill_between(
    x, 0, density, where=distribution_revenues >= break_even_revenue,
    facecolor="white", edgecolor="#777777", linewidth=0.6,
    label=f"Nonnegative net profit: {profitable_probability:.2%}",
)
ax.plot(x, density, color="#252525", lw=2)

for threshold, color in [(minimum_revenue, "#b73535"), (break_even_revenue, "#997600")]:
    ax.vlines(threshold / million, 0, revenue_distribution.pdf(threshold) * million,
              color=color, ls="--", lw=1.4)
ax.annotate(
    f"Default threshold\n${minimum_revenue / million:,.2f}m",
    xy=(minimum_revenue / million, revenue_distribution.pdf(minimum_revenue) * million),
    xytext=(0.07, 0.43), textcoords="axes fraction", color="#b73535",
    arrowprops={"arrowstyle": "->", "color": "#b73535"},
)
ax.annotate(
    f"Break-even revenue\n${break_even_revenue / million:,.2f}m",
    xy=(break_even_revenue / million, revenue_distribution.pdf(break_even_revenue) * million),
    xytext=(0.28, 0.73), textcoords="axes fraction", ha="center", color="#846600",
    arrowprops={"arrowstyle": "->", "color": "#997600"},
)
ax.set_title(f"{TICKER}: calibrated revenue distribution", pad=35)
ax.text(
    0.5, 1.025,
    f"Mean revenue: ${revenue_mean / million:,.2f}m  |  "
    f"Expected defaults: {expected_default_paths:,.0f} of {planned_path_count:,}",
    transform=ax.transAxes, ha="center", fontsize=10,
)
ax.set_xlabel("Revenue (USD millions)")
ax.set_ylabel("Probability density (per USD million)")
ax.set_xlim(x.min(), x.max())
ax.set_ylim(0, max(float(density.max()) * 1.12, 1.0 if revenue_std == 0 else 1e-12))
if revenue_std == 0:
    ax.axvline(revenue_mean / million, color="#252525", linewidth=2, label="Constant revenue: zero observed default rate")
ax.set_facecolor("white")
ax.ticklabel_format(axis="x", style="plain", useOffset=False)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="x", alpha=0.12)
ax.set_axisbelow(True)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=3, frameon=False)
fig.tight_layout()
plt.show()


### Sample revenue paths

Each of `N_PATHS` rows contains one sampled annual revenue. The seed `42` makes the sample repeatable. Existing model objects are closed before the sample is replaced on a rerun.

The financial seed remains 42 to isolate the daily-price dependence change.
It can still create shared standardized financial ranks across companies; this is
recorded separately in export metadata and is not a calibrated financial correlation model.


In [ ]:
# Close previous path models before replacing the dataframe on a rerun.
if "paths_df" in globals() and "income_statement" in paths_df:
    for old_model in paths_df["income_statement"]:
        old_model.close()
rng = np.random.default_rng(42)
paths_df = pd.DataFrame({
    "revenue": rng.normal(loc=revenue_distribution.mean, scale=revenue_distribution.stdev, size=N_PATHS)
})
paths_df


### Income statement for each path

Each row receives its own operational modelx income statement in the `income_statement` column. We set total revenue from that row's `revenue` value. Expenses stay at the original financial-statement amounts, while income tax and net profit recalculate.

Each model uses `FINANCIAL_YEAR` and the same preloaded financial record, while its account values remain independent of all other paths.


In [ ]:
def create_path_income_statement(revenue):
    model = generate_operational_income_statement(
        FINANCIAL_YEAR, financials=financials, eps_settings=valuation_settings
    )
    model.IncomeStatement.revenue.value = float(revenue)
    return model


# Close the old objects before replacing them when this cell is rerun.
if "income_statement" in paths_df.columns:
    for model in paths_df["income_statement"]:
        model.close()

paths_df["income_statement"] = paths_df["revenue"].apply(
    create_path_income_statement
)

paths_df


### Revenue and net profit from the models

Both histograms read their values directly from the modelx objects. The x-axes show revenue or net profit in USD millions, and each bar shows the number of paths in that range. Negative net profit means a loss.

Net-profit colors: **red** means the loss exhausts or exceeds cash reserves, **orange** means a loss covered by cash with some cash remaining, and **blue** means nonnegative profit.


In [ ]:
model_revenues = paths_df["income_statement"].map(
    lambda model: float(model.IncomeStatement.revenue())
)
model_net_profits = paths_df["income_statement"].map(
    lambda model: float(model.IncomeStatement.net_income())
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].hist(
    model_revenues / 1_000_000,
    bins=50, color="#477aa4", edgecolor="white", linewidth=0.6,
)
axes[0].set_title("Total revenue from the models")
axes[0].set_xlabel("Total revenue (USD millions)")

profit_values = model_net_profits.to_numpy()
cash_exhausted = profit_values <= -cash_amount
cash_covered = (profit_values > -cash_amount) & (profit_values < 0)
nonnegative_profit = profit_values >= 0

# Add the cash and zero-profit boundaries so bins do not span the thresholds.
profit_bins = np.unique(np.r_[
    np.histogram_bin_edges(profit_values / 1_000_000, bins=50),
    -cash_amount / 1_000_000, 0.0,
])
axes[1].hist(
    [profit_values[cash_exhausted] / 1_000_000,
     profit_values[cash_covered] / 1_000_000,
     profit_values[nonnegative_profit] / 1_000_000],
    bins=profit_bins, stacked=True,
    color=["#d9534f", "#f39c35", "#477aa4"],
    edgecolor="white", linewidth=0.6,
    label=[f"Cash exhausted: {cash_exhausted.sum():,}",
           f"Loss covered by cash: {cash_covered.sum():,}",
           f"Nonnegative profit: {nonnegative_profit.sum():,}"],
)
axes[1].axvline(-cash_amount / 1_000_000, color="#d9534f", ls="--", lw=1)
axes[1].axvline(0, color="#64748b", ls="--", lw=1)
axes[1].set_title("Net profit from the models")
axes[1].set_xlabel("Net profit (USD millions)")
axes[1].legend(loc="upper right", frameon=True, facecolor="white", edgecolor="none", framealpha=0.95, fontsize=8)

for ax in axes:
    ax.set_ylabel("Number of paths")
    ax.ticklabel_format(axis="both", style="plain", useOffset=False)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", alpha=0.18)
    ax.set_axisbelow(True)

fig.suptitle(f"{TICKER}: income statements for {len(paths_df):,} paths")
fig.tight_layout()
plt.show()


## Oil-price shock impacts

At each event, we calculate the change in the annual account amount at the current conditions:

`annualized delta = current annualized account × exposure × (1 − hedge coverage) × shock change`

The change takes effect on the event date. We add up the amount earned in each interval:

`annual account amount = sum(annualized account in each interval × interval days / simulation days)`

For one shock, this equals the starting account plus the delta multiplied by the remaining days divided by 365. For multiple shocks, the next change uses the latest annualized account amount. This preserves the amount earned before each event and compounds later changes correctly.

Hedge coverage is selected for each event date and converted from percent to a decimal. Negative shocks reduce the annualized account. Events before the simulation start or on/after its end are excluded. We keep the underlying oil price constant and use the supplied shock changes directly.


In [ ]:
def select_simulation_events(events, simulation_start, simulation_end):
    """Keep events in [start, end), ordered by calendar date."""
    start = pd.Timestamp(simulation_start).normalize()
    end = pd.Timestamp(simulation_end).normalize()
    if pd.isna(start) or pd.isna(end) or end <= start:
        raise ValueError("The simulation needs valid dates and a positive duration.")
    selected = events.copy()
    selected["date"] = pd.to_datetime(selected["date"]).dt.normalize()
    if selected["date"].isna().any():
        raise ValueError("Shock dates must be valid dates.")
    selected = selected.loc[(selected["date"] >= start) & (selected["date"] < end)]
    return selected.sort_values("date", kind="stable")


def calculate_time_weighted_account_value(
    account_value, events, simulation_start, simulation_end, calculate_delta
):
    """Sum each interval's annualized account amount times its share of the year."""
    start = pd.Timestamp(simulation_start).normalize()
    end = pd.Timestamp(simulation_end).normalize()
    if pd.isna(start) or pd.isna(end) or end <= start:
        raise ValueError("The simulation needs valid dates and a positive duration.")
    duration_days = (end - start).days
    annualized_value = float(account_value)
    if not np.isfinite(annualized_value):
        raise ValueError("The initial account value must be finite.")
    weighted_value = 0.0
    previous_date = start

    for event in events:
        event_date = pd.Timestamp(event["date"]).normalize()
        if pd.isna(event_date) or not previous_date <= event_date < end:
            raise ValueError("Shock events must be ordered and inside the simulation period.")
        interval_days = (event_date - previous_date).days
        weighted_value += annualized_value * (interval_days / duration_days)
        delta = calculate_delta(
            annualized_value, event["exposure"] * event.get("sensitivity", 1.0),
            event["hedge_coverage_pct"] / 100, event["price_change"],
        )
        annualized_value += delta
        if not all(np.isfinite(value) for value in [weighted_value, delta, annualized_value]):
            raise ValueError("The time-weighted shock calculation must remain finite.")
        previous_date = event_date

    weighted_value += annualized_value * ((end - previous_date).days / duration_days)
    if not np.isfinite(weighted_value):
        raise ValueError("The time-weighted account value must be finite.")
    return weighted_value


def calculate_oil_shock_delta(account_value, exposure, hedge_coverage, price_change):
    """Return the annualized delta at an event; coverage and change are decimals."""
    return account_value * exposure * (1 - hedge_coverage) * price_change


def build_oil_shock_schedule(
    events, hedges, company_information, simulation_start, simulation_end
):
    accounts = company_information["shock_impacts"]["oil_price_shock"]["accounts"]
    account_names = [item["account"] for item in accounts]
    if len(account_names) != len(set(account_names)):
        raise ValueError("Each oil-shock account must appear only once in the configuration.")

    oil_events = events.loc[events["shock_type"] == "oil_price_shock"].copy()
    oil_events = select_simulation_events(
        oil_events, simulation_start, simulation_end
    )
    starts = pd.to_datetime(hedges["hedge_coverage_period_start"]).dt.normalize()
    ends = pd.to_datetime(hedges["hedge_coverage_period_end"]).dt.normalize()

    rows = []
    for event in oil_events.itertuples(index=False):
        active_hedges = hedges.loc[(starts <= event.date) & (event.date <= ends)]
        if len(active_hedges) != 1:
            raise ValueError(f"Expected one oil hedge period for {event.date.date()}, found {len(active_hedges)}.")
        coverage_pct = float(active_hedges.iloc[0]["approximate_oil_hedge_coverage_pct"])
        price_change = float(event.price_change)
        if not np.isfinite(coverage_pct) or coverage_pct < 0 or not np.isfinite(price_change):
            raise ValueError("Hedge coverage must be nonnegative and all shock inputs must be finite.")

        for item in accounts:
            exposure = float(item["exposure"])
            sensitivity = float(item.get("sensitivity", 1.0))
            if not np.isfinite(sensitivity):
                raise ValueError("Account sensitivity must be finite.")
            if not np.isfinite(exposure) or exposure < 0:
                raise ValueError("Account exposure must be finite and nonnegative.")
            rows.append({
                "date": event.date,
                "account": item["account"],
                "exposure": exposure,
                "sensitivity": sensitivity,
                "hedge_coverage_pct": coverage_pct,
                "price_change": price_change,
            })

    return pd.DataFrame(rows, columns=[
        "date", "account", "exposure", "sensitivity", "hedge_coverage_pct", "price_change",
    ])


oil_shock_schedule_df = build_oil_shock_schedule(
    shock_events_df, oil_hedges_df, company_info,
    SIMULATION_START_DATE, SIMULATION_END_DATE,
)
oil_shock_schedule_df


### Apply the oil shocks to every path

The calculation splits the simulation at each event date and stores the resulting annual totals directly in `paths_df["income_statement"]`. Total revenue, income tax, and net profit recalculate automatically; other expense inputs stay fixed.

To apply the shocks again from the beginning, first rerun the cell that creates the path income statements. This prevents applying the same events twice.


In [ ]:
def resolve_shock_account(model, account_path):
    parts = account_path.split(".")
    if len(parts) < 2 or parts[0] != "income_statement" or any(
        not part.isidentifier() or part.startswith("_") for part in parts
    ):
        raise ValueError(f"Invalid income-statement account path: {account_path}")
    account = model.IncomeStatement
    try:
        for part in parts[1:]:
            account = getattr(account, part)
    except AttributeError as error:
        raise ValueError(f"Unknown shock account: {account_path}") from error
    # The total-revenue property is an intentional editable proxy: its setter
    # distributes the new amount over category inputs (CVX config targets it).
    if account_path == "income_statement.revenue" and hasattr(type(account), "value"):
        return account
    if not hasattr(account, "is_input") or not account.is_input():
        raise ValueError(f"Shock target must be an editable account input: {account_path}")
    return account


def apply_oil_shocks(paths, schedule, simulation_start, simulation_end):
    """Store each account's time-weighted annual amount in the path models."""
    if schedule.empty:
        return
    schedule = select_simulation_events(schedule, simulation_start, simulation_end)
    if schedule.empty:
        return
    models = paths["income_statement"]
    if not paths.index.is_unique or len({id(model) for model in models}) != len(models):
        raise ValueError("Each path must have a unique index and its own income statement.")
    if any(getattr(model, "oil_shocks_applied", False) for model in models):
        raise ValueError("Oil shocks were already applied. Rerun the path-income-statement creation cell first.")

    account_events = {
        name: group.to_dict("records")
        for name, group in schedule.groupby("account", sort=False, dropna=False)
    }
    assignments = []

    # Validate every account and calculate its annual total before updating any model.
    for model in models:
        for account_name, events in account_events.items():
            account = resolve_shock_account(model, account_name)
            after = calculate_time_weighted_account_value(
                account.value, events, simulation_start, simulation_end,
                calculate_oil_shock_delta,
            )
            assignments.append((account, after))

    for account, after in assignments:
        account.value = after
    for model in models:
        model.oil_shocks_applied = True


apply_oil_shocks(
    paths_df, oil_shock_schedule_df,
    SIMULATION_START_DATE, SIMULATION_END_DATE,
)


### Revenue and net profit after the oil shocks

Both histograms show the time-weighted annual results and read the updated values from each path's income statement. Each path is counted once. The left chart shows revenue, and the right chart shows net profit. Amounts are in USD millions, and bar heights show the number of paths.

Net-profit colors: **red** means the loss exhausts or exceeds cash reserves, **orange** means a loss covered by cash with some cash remaining, and **blue** means nonnegative profit.


In [ ]:
post_shock_revenues = paths_df["income_statement"].map(
    lambda model: float(model.IncomeStatement.revenue())
)
post_shock_net_profits = paths_df["income_statement"].map(
    lambda model: float(model.IncomeStatement.net_income())
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].hist(
    post_shock_revenues / 1_000_000,
    bins=50, color="#477aa4", edgecolor="white", linewidth=0.6,
)
axes[0].set_title("Revenue")
axes[0].set_xlabel("Revenue (USD millions)")

profit_values = post_shock_net_profits.to_numpy()
cash_exhausted = profit_values <= -cash_amount
cash_covered = (profit_values > -cash_amount) & (profit_values < 0)
nonnegative_profit = profit_values >= 0

# Add the cash and zero-profit boundaries so bins do not span the thresholds.
profit_bins = np.unique(np.r_[
    np.histogram_bin_edges(profit_values / 1_000_000, bins=50),
    -cash_amount / 1_000_000, 0.0,
])
axes[1].hist(
    [profit_values[cash_exhausted] / 1_000_000,
     profit_values[cash_covered] / 1_000_000,
     profit_values[nonnegative_profit] / 1_000_000],
    bins=profit_bins, stacked=True,
    color=["#d9534f", "#f39c35", "#477aa4"],
    edgecolor="white", linewidth=0.6,
    label=[f"Cash exhausted: {cash_exhausted.sum():,}",
           f"Loss covered by cash: {cash_covered.sum():,}",
           f"Nonnegative profit: {nonnegative_profit.sum():,}"],
)
axes[1].axvline(-cash_amount / 1_000_000, color="#d9534f", ls="--", lw=1)
axes[1].axvline(0, color="#64748b", ls="--", lw=1)
axes[1].set_title("Net profit")
axes[1].set_xlabel("Net profit (USD millions)")
axes[1].legend(loc="upper right", frameon=True, facecolor="white", edgecolor="none", framealpha=0.95, fontsize=8)

for ax in axes:
    ax.set_ylabel("Number of paths")
    ax.ticklabel_format(axis="both", style="plain", useOffset=False)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", alpha=0.18)
    ax.set_axisbelow(True)

fig.suptitle(f"{TICKER}: after oil-price shocks ({len(post_shock_revenues):,} paths)")
fig.tight_layout()
plt.show()


## Interest-rate shock impacts

For each configured interest-rate account, use the calendar-time calculation:

`annualized delta = current annualized account × exposure × (1 − applied protective coverage) × shock change`

The configured FY2025 exposure proxy includes floating/repricing debt and identified fixed-to-floating swaps before protective hedges. The source interest-hedge ratios have different meanings: ABT, COST, CZR, EOG, ITW, NOG, TXN and UAL use protective notional divided by floating debt; the other 30 series combine both swap directions divided by total borrowing. Gross mixed-direction ratios cannot reduce the exposed expense: pay-floating swaps create exposure already included in the configured proxy. For those series, retain the source ratio as `reported_hedge_notional_pct`, use `audit_only` basis, and apply zero additional protective coverage. This does not assert that no protective hedges exist. Some source zeros explicitly stand for undefined coverage.

The eight audited protective series default to `protective_fraction_of_floating_exposure`. An explicit `hedge_coverage_basis` under the company's `shock_impacts.interest_rate_shock` configuration can select either basis when a different scenario interpretation is supported. The schedule records the basis, source ratio and applied `hedge_coverage_pct` separately; no hedge amounts or directions are inferred from free-form source notes.

For one event, only `remaining calendar days / simulation days` of the delta enters the annual expense. Multiple events compound changes to the latest annualized account amount over their separate intervals. Applied coverage comes from the event-date period. A shock of `0.07` increases the exposed, unprotected expense by 7% from that date onward; a negative shock reduces it. This remains a proportional expense scenario using a funded-debt exposure proxy, not a complete debt coupon or bank asset/liability repricing model. Only events inside the simulation period are applied.


In [ ]:
def calculate_interest_rate_shock_delta(account_value, exposure, hedge_coverage, price_change):
    """Return the annualized delta at an event; coverage and change are decimals."""
    return account_value * exposure * (1 - hedge_coverage) * price_change


def build_interest_rate_shock_schedule(
    events, hedges, company_information, simulation_start, simulation_end
):
    accounts = company_information["shock_impacts"]["interest_rate_shock"]["accounts"]
    hedge_basis = notebook_support["interest_hedge_basis"](company_information)
    account_names = [item["account"] for item in accounts]
    if len(account_names) != len(set(account_names)):
        raise ValueError("Each interest-rate shock account must appear only once in the configuration.")

    interest_rate_events = events.loc[events["shock_type"] == "interest_rate_shock"].copy()
    interest_rate_events = select_simulation_events(
        interest_rate_events, simulation_start, simulation_end
    )
    starts = pd.to_datetime(hedges["hedge_coverage_period_start"]).dt.normalize()
    ends = pd.to_datetime(hedges["hedge_coverage_period_end"]).dt.normalize()

    rows = []
    for event in interest_rate_events.itertuples(index=False):
        active_hedges = hedges.loc[(starts <= event.date) & (event.date <= ends)]
        if len(active_hedges) != 1:
            raise ValueError(f"Expected one interest-rate hedge period for {event.date.date()}, found {len(active_hedges)}.")
        coverage_pct = float(active_hedges.iloc[0]["approximate_interest_rate_hedge_coverage_pct"])
        price_change = float(event.price_change)
        if not np.isfinite(coverage_pct) or coverage_pct < 0 or not np.isfinite(price_change):
            raise ValueError("Hedge coverage must be nonnegative and all shock inputs must be finite.")

        applied_coverage_pct = (
            coverage_pct if hedge_basis == "protective_fraction_of_floating_exposure" else 0.0
        )
        for item in accounts:
            exposure = float(item["exposure"])
            sensitivity = float(item.get("sensitivity", 1.0))
            if not np.isfinite(sensitivity):
                raise ValueError("Account sensitivity must be finite.")
            if not np.isfinite(exposure) or exposure < 0:
                raise ValueError("Account exposure must be finite and nonnegative.")
            rows.append({
                "date": event.date,
                "account": item["account"],
                "exposure": exposure,
                "sensitivity": sensitivity,
                "reported_hedge_notional_pct": coverage_pct,
                "hedge_coverage_basis": hedge_basis,
                "hedge_coverage_pct": applied_coverage_pct,
                "price_change": price_change,
            })

    return pd.DataFrame(rows, columns=[
        "date", "account", "exposure", "sensitivity", "reported_hedge_notional_pct",
        "hedge_coverage_basis", "hedge_coverage_pct", "price_change",
    ])


interest_rate_shock_schedule_df = build_interest_rate_shock_schedule(
    shock_events_df, interest_rate_hedges_df, company_info,
    SIMULATION_START_DATE, SIMULATION_END_DATE,
)
interest_rate_shock_schedule_df


### Apply the interest-rate shocks to every path

The time-weighted annual expense is stored directly in the configured account of each model in `paths_df["income_statement"]`. Income tax and net profit recalculate automatically.

To start again, recreate the path income statements and rerun the oil and interest-rate shock sections. A check prevents applying the interest-rate events twice to the same objects.


In [ ]:
def apply_interest_rate_shocks(paths, schedule, simulation_start, simulation_end):
    """Store each account's time-weighted annual amount in the path models."""
    if schedule.empty:
        return
    schedule = select_simulation_events(schedule, simulation_start, simulation_end)
    if schedule.empty:
        return
    models = paths["income_statement"]
    if not paths.index.is_unique or len({id(model) for model in models}) != len(models):
        raise ValueError("Each path must have a unique index and its own income statement.")
    if any(getattr(model, "interest_rate_shocks_applied", False) for model in models):
        raise ValueError("Interest-rate shocks were already applied. Rerun the path-income-statement creation cell first.")

    account_events = {
        name: group.to_dict("records")
        for name, group in schedule.groupby("account", sort=False, dropna=False)
    }
    assignments = []

    # Validate every account and calculate its annual total before updating any model.
    for model in models:
        for account_name, events in account_events.items():
            account = resolve_shock_account(model, account_name)
            after = calculate_time_weighted_account_value(
                account.value, events, simulation_start, simulation_end,
                calculate_interest_rate_shock_delta,
            )
            assignments.append((account, after))

    for account, after in assignments:
        account.value = after
    for model in models:
        model.interest_rate_shocks_applied = True


apply_interest_rate_shocks(
    paths_df, interest_rate_shock_schedule_df,
    SIMULATION_START_DATE, SIMULATION_END_DATE,
)


### Update each path's balance sheet and flag cash exhaustion

`update_balance_sheet` in `NOG.py` starts from an independent copy of the selected
year's opening balance sheet and applies the path's reported net profit after
all oil and interest-rate shocks. This model explicitly uses **net profit as the
change in cash**. No separate non-cash adjustments, investment, financing,
dividends or other balance-sheet movements are assumed.

Cash, current assets, total assets, retained earnings, parent equity and total
equity each increase by net profit, or decrease by a net loss. Liabilities,
non-cash assets and shares outstanding stay fixed. EPS normalization adjustments
never change the balance sheet.

`paths_df["balance_sheet"]` stores the projected balance sheet for each path;
`balance_sheet_audit_df` records its cash movements and accounting reconciliation.
Each call projects from the original financial record, so rerunning does not add
the same profit twice. Source rounding differences are preserved and reported.

A path defaults when projected cash is zero or negative. Under this assumption,
`ending_cash = opening_cash + net_profit <= 0` is the same rule as the previous
`net_profit <= -cash_amount` test. Negative projected cash is retained as an
unfunded requirement, with its magnitude in `funding_shortfall`; no new borrowing
is invented. Default dates are still randomly assigned by the price generator.


In [ ]:
update_balance_sheet = accounting_functions["update_balance_sheet"]
closing_balance_date = (SIMULATION_END_DATE - pd.Timedelta(days=1)).date().isoformat()
balance_sheet_projections = paths_df["income_statement"].map(
    lambda model: update_balance_sheet(
        financials, model=model, as_of_date=closing_balance_date
    )
)
paths_df["balance_sheet"] = balance_sheet_projections.map(lambda result: result["balance_sheet"])
balance_sheet_audit_df = pd.DataFrame(
    [{key: value for key, value in result.items() if key != "balance_sheet"}
     for result in balance_sheet_projections],
    index=paths_df.index,
)
final_net_profits = balance_sheet_audit_df["net_profit"]
paths_df["cash_end"] = balance_sheet_audit_df["ending_cash"]
paths_df["funding_shortfall"] = balance_sheet_audit_df["funding_shortfall"]
paths_df["is_default"] = balance_sheet_audit_df["is_default"]

print(f"Projected {len(paths_df):,} balance sheets using reported net profit as the cash change.")
print(
    f"Defaulting paths after all income-statement shocks: "
    f"{paths_df['is_default'].sum():,} / {len(paths_df):,}"
)
paths_df


### Revenue and net profit after the oil and interest-rate shocks

Both histograms show the time-weighted annual results and read the updated income-statement objects, counting each path once. The left chart shows total revenue; the right chart shows net profit. Amounts are in USD millions.

Net-profit colors: **red** means the loss exhausts or exceeds cash reserves, **orange** means the loss is covered by cash with some remaining, and **blue** means nonnegative profit.


In [ ]:
post_interest_shock_revenues = paths_df["income_statement"].map(
    lambda model: float(model.IncomeStatement.revenue())
)
post_interest_shock_net_profits = paths_df["income_statement"].map(
    lambda model: float(model.IncomeStatement.net_income())
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].hist(
    post_interest_shock_revenues / 1_000_000,
    bins=50, color="#477aa4", edgecolor="white", linewidth=0.6,
)
axes[0].set_title("Revenue")
axes[0].set_xlabel("Revenue (USD millions)")

profit_values = post_interest_shock_net_profits.to_numpy()
cash_exhausted = profit_values <= -cash_amount
cash_covered = (profit_values > -cash_amount) & (profit_values < 0)
nonnegative_profit = profit_values >= 0

# Add the cash and zero-profit boundaries so bins do not span the thresholds.
profit_bins = np.unique(np.r_[
    np.histogram_bin_edges(profit_values / 1_000_000, bins=50),
    -cash_amount / 1_000_000, 0.0,
])
axes[1].hist(
    [profit_values[cash_exhausted] / 1_000_000,
     profit_values[cash_covered] / 1_000_000,
     profit_values[nonnegative_profit] / 1_000_000],
    bins=profit_bins, stacked=True,
    color=["#d9534f", "#f39c35", "#477aa4"],
    edgecolor="white", linewidth=0.6,
    label=[f"Cash exhausted: {cash_exhausted.sum():,}",
           f"Loss covered by cash: {cash_covered.sum():,}",
           f"Nonnegative profit: {nonnegative_profit.sum():,}"],
)
axes[1].axvline(-cash_amount / 1_000_000, color="#d9534f", ls="--", lw=1)
axes[1].axvline(0, color="#64748b", ls="--", lw=1)
axes[1].set_title("Net profit")
axes[1].set_xlabel("Net profit (USD millions)")
axes[1].legend(loc="upper right", frameon=True, facecolor="white", edgecolor="none", framealpha=0.95, fontsize=8)

for ax in axes:
    ax.set_ylabel("Number of paths")
    ax.ticklabel_format(axis="both", style="plain", useOffset=False)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", alpha=0.18)
    ax.set_axisbelow(True)

fig.suptitle(f"{TICKER}: after oil-price and interest-rate shocks ({len(post_interest_shock_revenues):,} paths)")
fig.tight_layout()
plt.show()


### Historical stock reactions to WTI changes

Align WTI and stock closing-price levels to consecutive NYSE sessions before computing returns. A missing quote or a nonpositive endpoint makes that interval unavailable; never bridge a missing market session. The event sample uses finite WTI simple returns greater than 5% in absolute value and at most 100%, paired with the stock return over exactly the same interval. Use only event dates in the eight years before the simulation start, retaining the preceding session for the first interval.


In [ ]:
def prepare_oil_stock_reactions(stock_prices, wti_prices, session_dates, history_start, history_end, threshold=0.05):
    """Pair WTI and stock returns over the same consecutive NYSE sessions."""
    dates = pd.DatetimeIndex(session_dates)
    if dates.tz is not None or dates.has_duplicates or not dates.is_monotonic_increasing:
        raise ValueError("Historical session dates must be unique, ordered, and timezone-free.")
    start, end = pd.Timestamp(history_start), pd.Timestamp(history_end)
    if not start < end or not np.isfinite(threshold) or not 0 <= threshold < 1:
        raise ValueError("Historical dates and the WTI event threshold are invalid.")
    dates = dates[dates < end]
    levels = pd.DataFrame(index=dates)
    for source, date_column, value_column, output_column in (
        (wti_prices, "observation_date", "spot_price", "spot_price"),
        (stock_prices, "date", "close", "stock_close"),
    ):
        quotes = source[[date_column, value_column]].copy()
        quotes[date_column] = pd.to_datetime(quotes[date_column], errors="raise")
        quotes = quotes.loc[quotes[date_column] < end]
        if quotes[date_column].isna().any() or quotes[date_column].duplicated().any():
            raise ValueError("Historical quotes require unique, valid dates.")
        quotes[value_column] = pd.to_numeric(quotes[value_column], errors="coerce")
        levels[output_column] = quotes.set_index(date_column)[value_column].reindex(dates)

    for price_column, return_column in (("spot_price", "wti_return"), ("stock_close", "stock_return")):
        previous = levels[price_column].shift(1)
        positive_pair = (
            np.isfinite(levels[price_column]) & np.isfinite(previous)
            & levels[price_column].gt(0) & previous.gt(0)
        )
        levels[return_column] = (levels[price_column] / previous - 1).where(positive_pair)
    history = levels.loc[levels.index >= start].rename_axis("date").reset_index()
    large_moves = (
        np.isfinite(history["wti_return"]) & np.isfinite(history["stock_return"])
        & history["wti_return"].abs().gt(threshold)
        & history["wti_return"].between(-1.0, 1.0, inclusive="both")
    )
    reactions = history.loc[large_moves, ["date", "wti_return", "stock_return"]].reset_index(drop=True)
    return history, reactions


oil_reaction_lookback_years = 8
historical_oil_shock_threshold = 0.05
history_start_date = SIMULATION_START_DATE - pd.DateOffset(years=oil_reaction_lookback_years)
history_end_date = SIMULATION_START_DATE
# Include earlier sessions so the first event can use its actual prior-session quote.
historical_oil_sessions = market_calendar.valid_days(
    start_date=history_start_date - pd.DateOffset(months=1),
    end_date=history_end_date - pd.Timedelta(days=1),
).tz_localize(None)
wti_returns_df, oil_stock_reactions_df = prepare_oil_stock_reactions(
    stock_prices_df, wti_prices_df, historical_oil_sessions,
    history_start_date, history_end_date, historical_oil_shock_threshold,
)
oil_stock_reactions_df


In [ ]:
fig, ax = plt.subplots(figsize=(16, 10))
ax.scatter(
    oil_stock_reactions_df["wti_return"] * 100,
    oil_stock_reactions_df["stock_return"] * 100,
    s=40, color="#477aa4", alpha=0.75, edgecolors="white", linewidths=0.5,
)
ax.axhline(0, color="#64748b", linewidth=1)
ax.axvline(0, color="#64748b", linewidth=1)
ax.set_xlabel("WTI daily return (%)")
ax.set_ylabel(f"{TICKER} daily stock return (%)")
ax.set_title(
    f"WTI shocks and {TICKER} stock returns\n"
    f"{history_start_date:%d.%m.%Y}–{history_end_date - pd.Timedelta(days=1):%d.%m.%Y}"
    f" | {len(oil_stock_reactions_df):,} matched days",
    pad=14,
)
ax.xaxis.set_major_formatter("{x:.0f}%")
ax.yaxis.set_major_formatter("{x:.0f}%")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(alpha=0.18)
ax.set_axisbelow(True)
fig.tight_layout()

# Place every date near its point while keeping the labels apart.
fig.canvas.draw()
renderer = fig.canvas.get_renderer()
probe = ax.text(0, 0, "00.00.0000", fontsize=7, ha="center", va="center")
text_box = probe.get_window_extent(renderer)
label_width, label_height = text_box.width + 6, text_box.height + 4
probe.remove()

point_values = oil_stock_reactions_df[["wti_return", "stock_return"]].to_numpy() * 100
point_pixels = ax.transData.transform(point_values)
angles = np.linspace(0, 2 * np.pi, 24, endpoint=False)
radii = np.arange(14, 315, 10)
candidate_offsets = np.column_stack([
    (radii[:, None] * np.cos(angles)).ravel(),
    (radii[:, None] * np.sin(angles)).ravel(),
])
label_boxes = []
date_annotations = []
# Position the most crowded points first.
distances = np.linalg.norm(point_pixels[:, None] - point_pixels[None, :], axis=2)
label_order = np.argsort(-(distances < 80).sum(axis=1), kind="stable")

for index in label_order:
    centers = point_pixels[index] + candidate_offsets
    boxes = np.column_stack([
        centers[:, 0] - label_width / 2, centers[:, 1] - label_height / 2,
        centers[:, 0] + label_width / 2, centers[:, 1] + label_height / 2,
    ])
    fits = (
        (boxes[:, 0] >= ax.bbox.x0 + 3) & (boxes[:, 1] >= ax.bbox.y0 + 3)
        & (boxes[:, 2] <= ax.bbox.x1 - 3) & (boxes[:, 3] <= ax.bbox.y1 - 3)
    )
    overlap = np.zeros(len(boxes))
    if label_boxes:
        placed = np.asarray(label_boxes)
        overlap_width = np.maximum(0, np.minimum(boxes[:, None, 2], placed[:, 2]) - np.maximum(boxes[:, None, 0], placed[:, 0]))
        overlap_height = np.maximum(0, np.minimum(boxes[:, None, 3], placed[:, 3]) - np.maximum(boxes[:, None, 1], placed[:, 1]))
        overlap = (overlap_width * overlap_height).sum(axis=1)
    covers_point = (
        (point_pixels[None, :, 0] >= boxes[:, None, 0] - 4)
        & (point_pixels[None, :, 0] <= boxes[:, None, 2] + 4)
        & (point_pixels[None, :, 1] >= boxes[:, None, 1] - 4)
        & (point_pixels[None, :, 1] <= boxes[:, None, 3] + 4)
    ).sum(axis=1)
    scores = overlap * 1_000_000 + covers_point * 10_000 + np.linalg.norm(candidate_offsets, axis=1)
    scores[~fits] = np.inf
    best = int(np.argmin(scores))
    label_boxes.append(boxes[best])
    row = oil_stock_reactions_df.iloc[index]
    date_annotations.append(ax.annotate(
        row["date"].strftime("%d.%m.%Y"),
        xy=point_values[index],
        xytext=candidate_offsets[best] * 72 / fig.dpi,
        textcoords="offset points", fontsize=7, ha="center", va="center",
        color="#34495e",
        bbox={"boxstyle": "square,pad=0.1", "facecolor": "white", "edgecolor": "none", "alpha": 0.9},
        arrowprops={"arrowstyle": "-", "color": "#94a3b8", "lw": 0.5, "shrinkA": 2, "shrinkB": 3},
    ))

plt.show()


### WTI sensitivity: beta

We fit a straight line to the matched historical returns:

`stock return = wti_alpha + wti_beta × WTI return`

`wti_beta` measures how many percentage points the fitted stock return changes when the WTI return increases by one percentage point. `wti_alpha` is the line's intercept.

This estimates the relationship for the selected historical shock days. It is an average relationship, not an exact prediction for each event.


In [ ]:
wti_shock_returns = oil_stock_reactions_df["wti_return"].to_numpy(dtype=float)
stock_shock_returns = oil_stock_reactions_df["stock_return"].to_numpy(dtype=float)

if len(wti_shock_returns) < 2 or np.ptp(wti_shock_returns) == 0:
    raise ValueError("Calculating beta requires at least two observations with different WTI returns.")

# The regression slope is covariance divided by the variance of WTI returns.
wti_beta = float(
    np.cov(wti_shock_returns, stock_shock_returns, ddof=1)[0, 1]
    / np.var(wti_shock_returns, ddof=1)
)
wti_alpha = float(stock_shock_returns.mean() - wti_beta * wti_shock_returns.mean())

print(f"{TICKER} WTI beta: {wti_beta:.4f}")
print(f"Regression intercept (alpha): {wti_alpha:.4%}")


## Immediate stock-market reaction to interest-rate changes

We use `data/rates/us_interest_rates.csv`, which contains the [effective federal funds rate (DFF)](https://fred.stlouisfed.org/series/DFF) in percent. We select changes of at least **0.5 percentage points in absolute terms** during the eight years before `SIMULATION_START_DATE`. Both increases and decreases are included; changes of exactly ±0.5 percentage points are included.

Changes are measured in **percentage points**: a move from 4% to 5% is +1 percentage point. Each change is matched with the stock return on that same date, and each plotted point is labeled with its date. Dates without a stock return cannot be included in the comparison; the output shows how many were unmatched.

These are changes in the observed effective rate, using its recorded date. Each series uses its previous available observation, which can differ across the two series. Stock returns use the supplied closing prices.


In [ ]:
interest_reaction_lookback_years = 8
interest_rate_change_threshold_pp = 0.5
interest_history_start_date = SIMULATION_START_DATE - pd.DateOffset(years=interest_reaction_lookback_years)
interest_history_end_date = SIMULATION_START_DATE

interest_rates_df = pd.read_csv(DATA_DIR / "rates" / "us_interest_rates.csv")[[
    "date", "effective_federal_funds_rate_percent",
]].rename(columns={"effective_federal_funds_rate_percent": "interest_rate_percent"})
interest_rates_df["date"] = pd.to_datetime(interest_rates_df["date"])
interest_rates_df["interest_rate_percent"] = pd.to_numeric(
    interest_rates_df["interest_rate_percent"], errors="coerce"
)
interest_rates_df = (
    interest_rates_df.dropna(subset=["date", "interest_rate_percent"])
    .sort_values("date")
    .loc[lambda data: data["date"] < interest_history_end_date]
    .copy()
)

# Calculate differences before limiting dates, preserving the first day's prior rate.
interest_rates_df["interest_rate_change_pp"] = interest_rates_df["interest_rate_percent"].diff()
interest_rates_df = interest_rates_df.loc[
    interest_rates_df["date"] >= interest_history_start_date
].copy()
# Round only for comparison so exactly half-point moves are included reliably.
rate_change_days_df = interest_rates_df.loc[
    np.isfinite(interest_rates_df["interest_rate_change_pp"])
    & (interest_rates_df["interest_rate_change_pp"].abs().round(10) >= interest_rate_change_threshold_pp),
    ["date", "interest_rate_change_pp"],
].copy()

interest_stock_reactions_df = rate_change_days_df.merge(
    stock_prices_df[["date", "daily_simple_return"]].rename(columns={"daily_simple_return": "stock_return"}),
    on="date", how="inner", validate="one_to_one",
)
interest_stock_reactions_df = (
    interest_stock_reactions_df.loc[np.isfinite(interest_stock_reactions_df["stock_return"])]
    .sort_values("date")
    .reset_index(drop=True)
)

print(f"Selected interest-rate changes: {len(rate_change_days_df):,}")
print(f"Matched stock returns: {len(interest_stock_reactions_df):,}")
print(f"Unmatched dates: {len(rate_change_days_df) - len(interest_stock_reactions_df):,}")
interest_stock_reactions_df


In [ ]:
fig, ax = plt.subplots(figsize=(18, 12))
ax.scatter(
    interest_stock_reactions_df["interest_rate_change_pp"],
    interest_stock_reactions_df["stock_return"] * 100,
    s=40, color="#477aa4", alpha=0.75, edgecolors="white", linewidths=0.5,
)
ax.axhline(0, color="#64748b", linewidth=1)
ax.axvline(0, color="#64748b", linewidth=1)
ax.set_xlabel("Interest-rate change (percentage points)")
ax.set_ylabel(f"{TICKER} daily stock return (%)")
ax.set_title(
    f"Interest-rate changes and {TICKER} stock returns\n"
    f"{interest_history_start_date:%d.%m.%Y}–{interest_history_end_date - pd.Timedelta(days=1):%d.%m.%Y}"
    f" | {len(interest_stock_reactions_df):,} matched days",
    pad=14,
)
ax.xaxis.set_major_formatter("{x:.2f}")
ax.yaxis.set_major_formatter("{x:g}%")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(alpha=0.18)
ax.set_axisbelow(True)
fig.tight_layout()

# Place every date near its point while keeping the labels apart.
fig.canvas.draw()
renderer = fig.canvas.get_renderer()
probe = ax.text(0, 0, "00.00.0000", fontsize=7, ha="center", va="center")
text_box = probe.get_window_extent(renderer)
label_width, label_height = text_box.width + 6, text_box.height + 4
probe.remove()

point_values = np.column_stack([
    interest_stock_reactions_df["interest_rate_change_pp"].to_numpy(),
    interest_stock_reactions_df["stock_return"].to_numpy() * 100,
])
point_pixels = ax.transData.transform(point_values)
angles = np.linspace(0, 2 * np.pi, 24, endpoint=False)
radii = np.arange(14, 445, 10)
candidate_offsets = np.column_stack([
    (radii[:, None] * np.cos(angles)).ravel(),
    (radii[:, None] * np.sin(angles)).ravel(),
])
label_boxes = []
date_annotations = []
# Position the most crowded points first.
distances = np.linalg.norm(point_pixels[:, None] - point_pixels[None, :], axis=2)
label_order = np.argsort(-(distances < 80).sum(axis=1), kind="stable")

for index in label_order:
    centers = point_pixels[index] + candidate_offsets
    boxes = np.column_stack([
        centers[:, 0] - label_width / 2, centers[:, 1] - label_height / 2,
        centers[:, 0] + label_width / 2, centers[:, 1] + label_height / 2,
    ])
    fits = (
        (boxes[:, 0] >= ax.bbox.x0 + 3) & (boxes[:, 1] >= ax.bbox.y0 + 3)
        & (boxes[:, 2] <= ax.bbox.x1 - 3) & (boxes[:, 3] <= ax.bbox.y1 - 3)
    )
    overlap = np.zeros(len(boxes))
    if label_boxes:
        placed = np.asarray(label_boxes)
        overlap_width = np.maximum(0, np.minimum(boxes[:, None, 2], placed[:, 2]) - np.maximum(boxes[:, None, 0], placed[:, 0]))
        overlap_height = np.maximum(0, np.minimum(boxes[:, None, 3], placed[:, 3]) - np.maximum(boxes[:, None, 1], placed[:, 1]))
        overlap = (overlap_width * overlap_height).sum(axis=1)
    covers_point = (
        (point_pixels[None, :, 0] >= boxes[:, None, 0] - 4)
        & (point_pixels[None, :, 0] <= boxes[:, None, 2] + 4)
        & (point_pixels[None, :, 1] >= boxes[:, None, 1] - 4)
        & (point_pixels[None, :, 1] <= boxes[:, None, 3] + 4)
    ).sum(axis=1)
    scores = overlap * 1_000_000 + covers_point * 10_000 + np.linalg.norm(candidate_offsets, axis=1)
    scores[~fits] = np.inf
    best = int(np.argmin(scores))
    label_boxes.append(boxes[best])
    row = interest_stock_reactions_df.iloc[index]
    date_annotations.append(ax.annotate(
        row["date"].strftime("%d.%m.%Y"),
        xy=point_values[index],
        xytext=candidate_offsets[best] * 72 / fig.dpi,
        textcoords="offset points", fontsize=7, ha="center", va="center",
        color="#34495e",
        bbox={"boxstyle": "square,pad=0.1", "facecolor": "white", "edgecolor": "none", "alpha": 0.9},
        arrowprops={"arrowstyle": "-", "color": "#94a3b8", "lw": 0.5, "shrinkA": 2, "shrinkB": 3},
    ))

plt.show()


### Interest-rate sensitivity: beta

We fit a straight line to the matched rate-change days that pass the filter:

`stock return = interest_rate_alpha + interest_rate_beta × (interest-rate change in percentage points / 100)`

Dividing the percentage-point change by 100 gives a decimal rate change: +1 percentage point becomes +0.01. The beta measures the change in the fitted stock return, in percentage points, for a one-percentage-point rate increase. The alpha is the line's intercept.

We also report R², the share of stock-return variation explained by this line. A value near zero means the relationship is weak in this sample.


In [ ]:
# Use decimal units for both the rate change and the stock return.
interest_rate_changes = interest_stock_reactions_df["interest_rate_change_pp"].to_numpy(dtype=float) / 100
interest_stock_returns = interest_stock_reactions_df["stock_return"].to_numpy(dtype=float)

if len(interest_rate_changes) < 2 or np.ptp(interest_rate_changes) == 0:
    raise ValueError("Calculating beta requires at least two observations with different rate changes.")

interest_rate_beta = float(
    np.cov(interest_rate_changes, interest_stock_returns, ddof=1)[0, 1]
    / np.var(interest_rate_changes, ddof=1)
)
interest_rate_alpha = float(
    interest_stock_returns.mean() - interest_rate_beta * interest_rate_changes.mean()
)
interest_fitted_returns = interest_rate_alpha + interest_rate_beta * interest_rate_changes
interest_total_variation = np.sum((interest_stock_returns - interest_stock_returns.mean()) ** 2)
interest_rate_r_squared = (
    float(1 - np.sum((interest_stock_returns - interest_fitted_returns) ** 2) / interest_total_variation)
    if interest_total_variation > 0 else np.nan
)

print(f"{TICKER} interest-rate beta: {interest_rate_beta:.4f} (stock-return pp per 1 rate pp)")
print(f"Regression intercept (alpha): {interest_rate_alpha:.4%}")
print(f"R²: {interest_rate_r_squared:.4f}")


## Company-specific EPS and equity valuation

The company module calculates reported and normalized EPS separately. Only explicit normalization adjustments are applied; absent normalization means reported earnings. The configured `eps_field` and P/E are respected. NOG's explicit reference-price/normalized-EPS policy is retained. A missing P/E uses asset valuation throughout, rather than inventing a multiple for loss-making companies.

Cash projections use consolidated income; EPS may use parent earnings, preferred-dividend deductions, or participating share allocations. The notebook checks that projected consolidated and parent earnings still match the live models.

Asset valuation uses projected book assets less liabilities and the company's required preferred and minority claims. Sourced economic preferred claims replace par values where necessary. Actual shares outstanding are used when supplied; otherwise the audit identifies a weighted-average share-count proxy. Visa uses its as-converted denominator and retains converted preferred equity.

`raw_share_value_end` preserves the signed estimate. The explicit `limited_liability_zero` price policy floors negative equity estimates at zero without changing earnings, cash, or default flags. A zero book-based equity estimate is not proof that a listed stock has no market value. All these estimates remain scenario assumptions, not validated market-price forecasts.

For very severe losses, negative cash is an unfunded requirement. Asset valuation reclassifies that amount from negative gross assets to a senior funding claim, preserving signed net assets exactly. The projected accounts remain unchanged; this does not assume actual borrowing.


In [ ]:
valuation = valuation_settings
calculate_eps = accounting_functions["calculate_eps"]
reference_eps_audit = calculate_eps(financials, valuation, model=operational_income_statement_model)
reference_prices = stock_prices_df.loc[
    stock_prices_df["date"] < SIMULATION_START_DATE, ["date", "close"]
].copy().sort_values("date")
if reference_prices.empty:
    raise ValueError("A valid closing price before the simulation start is required.")
initial_price_date = reference_prices.iloc[-1]["date"]
initial_share_price = float(reference_prices.iloc[-1]["close"])
pe_policy = notebook_support["select_pe_policy"](valuation, reference_eps_audit, initial_share_price)
reference_eps = pe_policy["reference_eps"]
constant_pe_ratio = pe_policy["pe_ratio"]

eps_audit_df = pd.DataFrame(
    [calculate_eps(financials, valuation, model=model) for model in paths_df["income_statement"]],
    index=paths_df.index,
)
for column in [
    "reported_common_earnings", "reported_basic_eps", "reported_diluted_eps",
    "normalized_common_earnings", "normalized_basic_eps", "normalized_diluted_eps",
    "earnings_adjustment_before_tax", "earnings_adjustment_after_tax",
    "normalization_tax_rate", "pe_valuation_applicable",
]:
    paths_df[column] = eps_audit_df[column]
paths_df["eps"] = eps_audit_df[pe_policy["eps_column"]]
pe_mask = paths_df["eps"].gt(0) & (constant_pe_ratio is not None)

if "balance_sheet" not in paths_df or "balance_sheet_audit_df" not in globals():
    raise ValueError("Run the balance-sheet projection before valuation.")
if not balance_sheet_audit_df.index.equals(paths_df.index):
    raise ValueError("Balance sheets and income statements must use the same path index.")
statement_amount_scale = float(balance_sheet_audit_df["amount_scale"].iloc[0])
if not balance_sheet_audit_df["amount_scale"].eq(statement_amount_scale).all():
    raise ValueError("All path balance sheets must use the same monetary units.")
# Cash uses consolidated income, while EPS can use parent/common/two-class income.
current_consolidated_profit = paths_df["income_statement"].map(lambda model: float(model.IncomeStatement.net_income()))
current_parent_profit = paths_df["income_statement"].map(
    lambda model: float(model.IncomeStatement.net_income_attributable_to.parent_company_shareholders())
)
if not np.allclose(balance_sheet_audit_df["net_profit"], current_consolidated_profit, rtol=1e-12, atol=1e-5):
    raise ValueError("Consolidated earnings changed after projection; rerun that cell.")
if "parent_net_profit" in balance_sheet_audit_df and not np.allclose(
    balance_sheet_audit_df["parent_net_profit"], current_parent_profit, rtol=1e-12, atol=1e-5
):
    raise ValueError("Parent earnings changed after projection; rerun that cell.")
asset_valuation_inputs = pd.DataFrame([
    notebook_support["asset_valuation_inputs"](
        accounting_functions, financials, row["balance_sheet"], row["income_statement"], statement_amount_scale
    ) for _, row in paths_df.iterrows()
], index=paths_df.index)
asset_values_per_share = asset_valuation_inputs["value_per_share"]
paths_df["pe_valuation_product"] = paths_df["eps"] * (constant_pe_ratio if constant_pe_ratio is not None else np.nan)
if not np.isfinite(paths_df.loc[pe_mask, "pe_valuation_product"]).all():
    raise ValueError("Applicable P/E products must be finite.")
paths_df["asset_value_per_share"] = asset_values_per_share
paths_df["valuation_method"] = np.where(pe_mask, pe_policy["eps_column"] + "_pe", "balance_sheet_assets")
paths_df["raw_share_value_end"] = paths_df["pe_valuation_product"].where(pe_mask, asset_values_per_share)
if not np.isfinite(paths_df["raw_share_value_end"]).all():
    raise ValueError("All raw valuation estimates must be finite.")
if NONPOSITIVE_ASSET_VALUE_POLICY != "limited_liability_zero":
    raise ValueError("Choose the documented limited_liability_zero equity-price policy.")
# Keep signed book value in the audit; limited-liability equity prices cannot be negative.
# A zero equity valuation is distinct from cash exhaustion and does not change is_default.
paths_df["share_price_start"] = initial_share_price
paths_df["share_price_end"] = paths_df["raw_share_value_end"].clip(lower=0.0)
paths_df["zero_equity_value"] = paths_df["share_price_end"].eq(0)
paths_df["valuation_target_positive"] = paths_df["share_price_end"].gt(0)
for column in ["asset_value_per_share", "valuation_method", "raw_share_value_end", "share_price_end", "zero_equity_value"]:
    eps_audit_df[column] = paths_df[column]
eps_audit_df["asset_estimate"] = asset_valuation_inputs["estimated_asset_value"]
eps_audit_df["asset_liabilities"] = asset_valuation_inputs["liabilities"]
eps_audit_df["asset_shares_outstanding"] = asset_valuation_inputs["shares_outstanding"]
print(f"Reference reported diluted EPS: {reference_eps_audit['reported_diluted_eps']:.6f}")
print(f"Reference valuation EPS ({pe_policy['eps_column']}): {reference_eps:.6f}")
print(f"Reference price ({initial_price_date:%Y-%m-%d}): ${initial_share_price:.2f}")
print(f"P/E policy: {pe_policy['method']}; multiple: {constant_pe_ratio}")
print(f"Valuation methods: {pe_mask.sum():,} P/E; {(~pe_mask).sum():,} asset fallback.")
print(f"Zero equity valuations: {paths_df['zero_equity_value'].sum():,}; cash defaults: {paths_df['is_default'].sum():,}")
paths_df[["reported_diluted_eps", "normalized_diluted_eps", "valuation_method",
          "is_default", "share_price_start", "raw_share_value_end", "share_price_end"]]


## Simulated equity-price targets

The histogram displays nonnegative equity-price targets after the declared limited-liability rule. Signed asset estimates remain in `asset_valuation_inputs`, `eps_audit_df`, and `raw_share_value_end`. Cash default and zero equity valuation are separate flags. Daily prices are stochastic; a cash-default flag can set them permanently to zero earlier in the period.


In [ ]:
simulated_share_prices = paths_df["share_price_end"].astype(float)
if simulated_share_prices.empty or not np.isfinite(simulated_share_prices).all():
    raise ValueError("Share valuations must be nonempty and finite; inspect eps_audit_df.")
print(f"P/E valuations: {pe_mask.sum():,}")
print(f"Balance-sheet asset fallbacks: {(~pe_mask).sum():,}")

initial_share_price = float(paths_df["share_price_start"].iloc[0])
average_simulated_price = float(simulated_share_prices.mean())
average_price_change = average_simulated_price - initial_share_price
average_price_change_pct = (average_simulated_price / initial_share_price - 1) * 100
paths_above_initial = int((simulated_share_prices > initial_share_price).sum())
fraction_above_initial = paths_above_initial / len(simulated_share_prices)

print(f"Initial price ({initial_price_date:%d.%m.%Y}): ${initial_share_price:.2f}")
print(f"Average share valuation: ${average_simulated_price:.2f}")
print(f"Average price change: {average_price_change:+.2f} USD ({average_price_change_pct:+.2f}%)")
print(f"Paths above initial price: {paths_above_initial:,} / {len(simulated_share_prices):,} ({fraction_above_initial:.2%})")

fig, ax = plt.subplots(figsize=(12, 6))
ax.hist(simulated_share_prices, bins=50, color="#477aa4", edgecolor="white", linewidth=0.6)
ax.axvline(
    initial_share_price, color="#555555", linestyle="--", linewidth=2,
    label=f"Initial price: ${initial_share_price:.2f}",
)
ax.axvline(
    average_simulated_price, color="#16856b", linewidth=2,
    label=f"Average share valuation: ${average_simulated_price:.2f}",
)
ax.set_title(f"{TICKER}: share valuations ({len(simulated_share_prices):,} paths)", fontsize=14, pad=38)
ax.text(
    0.5, 1.025,
    f"Mean change: {average_price_change:+.2f} USD ({average_price_change_pct:+.2f}%)"
    f"  |  Above initial price: {fraction_above_initial:.2%}",
    transform=ax.transAxes, ha="center", fontsize=10,
)
ax.set_xlabel("Estimated share price (USD)")
ax.set_ylabel("Number of paths")
ax.xaxis.set_major_formatter("${x:.0f}")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.18)
ax.set_axisbelow(True)
ax.legend(frameon=False)
fig.tight_layout()
plt.show()


## Historical daily volatility

We use the five years before `SIMULATION_START_DATE` and the `daily_log_return` column calculated when loading stock prices:

`log_return = ln(close / previous_close)`

Daily volatility is the sample standard deviation of these returns (`ddof=1`). It measures how much daily returns vary and is stored in `daily_log_volatility` as a decimal; for example, `0.03` means **3% per day**.

The return date must fall inside the five-year window. We use the preceding available close to calculate the first return, even if that close is just before the window. The supplied prices are unadjusted closing prices, so this measures price returns and excludes dividend income.


In [ ]:
volatility_lookback_years = 5
volatility_start_date = SIMULATION_START_DATE - pd.DateOffset(years=volatility_lookback_years)
volatility_end_date = SIMULATION_START_DATE

historical_stock_volatility_df = stock_prices_df.loc[
    stock_prices_df["date"] < volatility_end_date, ["date", "close", "daily_log_return"]
].sort_values("date").rename(columns={"daily_log_return": "log_return"}).copy()
historical_stock_volatility_df["previous_close"] = historical_stock_volatility_df["close"].shift(1)
historical_stock_volatility_df = historical_stock_volatility_df.loc[
    historical_stock_volatility_df["date"] >= volatility_start_date
].reset_index(drop=True)

if len(historical_stock_volatility_df) < 2:
    raise ValueError("Daily volatility requires at least two daily returns.")
if historical_stock_volatility_df["date"].duplicated().any():
    raise ValueError("Each trading date must have exactly one closing price.")
volatility_price_pairs = historical_stock_volatility_df[["close", "previous_close"]].to_numpy(dtype=float)
if not np.isfinite(volatility_price_pairs).all() or (volatility_price_pairs <= 0).any():
    raise ValueError("Log returns require valid, positive current and previous closing prices.")

if not np.isfinite(historical_stock_volatility_df["log_return"]).all():
    raise ValueError("All daily log returns in the volatility window must be finite.")
daily_log_volatility = float(historical_stock_volatility_df["log_return"].std(ddof=1))

print(f"{TICKER} daily log-return volatility: {daily_log_volatility:.4%}")
print(
    f"Trading dates: {historical_stock_volatility_df['date'].min():%d.%m.%Y} "
    f"to {historical_stock_volatility_df['date'].max():%d.%m.%Y}"
)
print(f"Daily log returns: {len(historical_stock_volatility_df):,}")


## Daily stock-price paths

We generate one closing price per actual NYSE trading day, using the calendar from the setup. A weekend or holiday shock moves to the next trading date inside the simulation. For 2026, `day_1` is 2 January and `day_251` is 31 December. `daily_price_dates` maps the column names to dates. The calendar comes from [pandas_market_calendars](https://pandas-market-calendars.readthedocs.io/en/latest/usage.html) and matches the [NYSE trading calendar](https://www.nyse.com/publicdocs/Trading_Days.pdf).

First, we calculate each immediate market shock using the existing linear models, including their intercepts. An oil event with exactly 0% change bypasses the calibrated regression and has a predicted stock return of 0%; its event row is retained for audit. Nonzero oil changes enter the regression as simple percentage changes. Interest changes are **relative increases in the reference rate**: a shock of `0.07` changes 3.64% to 3.8948%, giving a change of 0.2548 percentage points and a regression input of `0.002548`. The initial reference rate is the last observed rate before the simulation; later interest shocks use its updated level. If the converted absolute interest-rate change is exactly zero, the interest-rate regression is bypassed and the predicted stock return is 0%; the event row is retained for audit. Nonzero rate changes continue to use the calibrated regression, including its intercept.

The table below contains one row per market shock. Its predicted stock return is added to the normal daily movement as a separate price jump. These are market reactions; the existing expected end prices already include the long-term income-statement effects.


In [ ]:
reference_rates = interest_rates_df.loc[
    interest_rates_df["date"] < SIMULATION_START_DATE,
    ["date", "interest_rate_percent"],
].dropna(subset=["interest_rate_percent"]).sort_values("date")
if reference_rates.empty:
    raise ValueError("A reference interest rate before the simulation start is required.")
initial_reference_rate_percent = float(reference_rates.iloc[-1]["interest_rate_percent"])
reference_rate_percent = initial_reference_rate_percent
if not np.isfinite(reference_rate_percent):
    raise ValueError("The reference interest rate must be finite.")

market_events = select_simulation_events(
    shock_events_df, SIMULATION_START_DATE, SIMULATION_END_DATE
)
market_shock_rows = []
for event in market_events.itertuples(index=False):
    day_index = int(trading_dates.searchsorted(event.date, side="left"))
    if day_index == TRADING_DAYS:
        # The next market session is outside this simulation period.
        continue
    change = float(event.price_change)
    if not np.isfinite(change):
        raise ValueError("Market shock changes must be finite.")

    if event.shock_type == "oil_price_shock":
        regression_input = change
        # A zero oil change has no market jump; do not apply the fitted intercept.
        predicted_return = 0.0 if change == 0.0 else wti_alpha + wti_beta * regression_input
    elif event.shock_type == "interest_rate_shock":
        rate_change_pp = reference_rate_percent * change
        regression_input = rate_change_pp / 100
        # No actual rate change means no market jump, including at a zero reference rate.
        predicted_return = (
            0.0 if regression_input == 0.0
            else interest_rate_alpha + interest_rate_beta * regression_input
        )
        reference_rate_percent += rate_change_pp
        if not np.isfinite(reference_rate_percent):
            raise ValueError("The updated reference interest rate must be finite.")
    else:
        raise ValueError(f"Unsupported market shock type: {event.shock_type}")

    if not np.isfinite(predicted_return) or predicted_return <= -1:
        raise ValueError("A predicted stock return must be finite and greater than -100%.")
    market_shock_rows.append({
        "event_date": event.date,
        "trading_date": trading_dates[day_index],
        "day_number": day_index + 1,
        "shock_type": event.shock_type,
        "regression_input": float(regression_input),
        "predicted_stock_return": float(predicted_return),
    })

market_shocks_df = pd.DataFrame(market_shock_rows, columns=[
    "event_date", "trading_date", "day_number", "shock_type",
    "regression_input", "predicted_stock_return",
])
print(f"Reference interest rate before shocks: {initial_reference_rate_percent:.4f}%")
print(f"Reference interest rate after shocks: {reference_rate_percent:.4f}%")
market_shocks_df


### Generate daily prices

Positive targets use geometric Brownian motion with the target as the expected terminal price. Known market shocks compound on their scheduled trading days. Their total log jump is compensated once in the background drift, preserving the expected terminal target and full-horizon diffusion variance, including when a shock falls on the final session. Random paths are not reanchored after shocks and are not forced to match positive targets.

A zero target cannot be represented by GBM. Such paths use a stochastic positive reference process multiplied by the fraction of time remaining, reaching zero at the last session. This is an explicit zero-endpoint scenario convention. Cash-default paths independently become permanently zero at their sampled default day; a zero valuation alone never changes the cash-default flag.

Daily Gaussian innovations now come from the same immutable annual dataset for all
30 companies. Axes are simulation ID (0..N-1), actual NYSE trading date, and the
fixed company order. Each company reads its own slice; no company-local daily
random generator remains. All shock scenarios reuse the identical slice.

The preparation notebook estimates Pearson correlations using a common cohort of
consecutive-session daily log returns over the five calendar years strictly before
January 1 of the simulation year. It uses split-adjusted closes, no forward fill,
no gap bridging, and at least 500 common valid return dates. Regularization is
`R = 0.95 * sample_R + 0.05 * identity` by default. Cholesky generation preserves
standard-normal marginals and targets this regularized correlation, rather than
forcing sample correlations to equal it exactly. Innovations remain independent
across trading days. Existing company volatility, drift, valuation and shock rules
still apply, so realized stock-return correlations need not equal innovation correlations.

The shared price seed depends only on the master seed, simulation year and a
versioned price-stream identifier. It excludes company, scenario and model.
Regression integration is deferred: the dataset supports another consumer, but the
regression notebook still uses its existing randomness and must not be described
as sharing these innovations yet. The V2/ARCH/GARCH notebook is excluded.

Default timing still uses seed 2027 and the original default-row allocation order.
Its shared-seed dependence remains deliberately unchanged, alongside revenue seed 42.


In [ ]:
def generate_daily_stock_prices(
    paths, dates, market_shocks, daily_volatility, *, innovations, default_seed=2027
):
    """Use a verified shared Gaussian slice; preserve valuation and default timing."""
    dates = pd.DatetimeIndex(dates)
    n_days = len(dates)
    if n_days == 0 or not dates.is_unique or not dates.is_monotonic_increasing:
        raise ValueError("Trading dates must be nonempty, unique, and ordered.")
    if paths.empty or not paths.index.is_unique:
        raise ValueError("Paths must be nonempty and have a unique index.")
    if "is_default" not in paths.columns:
        raise ValueError("Flag defaulting paths after applying all income-statement shocks first.")
    default_flags = paths["is_default"]
    if not pd.api.types.is_bool_dtype(default_flags.dtype) or default_flags.isna().any():
        raise ValueError("Each path must have a True or False default flag.")
    default_rows = np.flatnonzero(default_flags.to_numpy(dtype=bool))
    endpoints = paths[["share_price_start", "share_price_end"]].to_numpy(dtype=float)
    if not np.isfinite(endpoints[:, 1]).all() or (endpoints[:, 1] < 0).any():
        raise ValueError("Equity-price targets must be finite and nonnegative.")
    zero_targets = endpoints[:, 1] == 0
    # Zero cannot be a lognormal mean. Use an explicit zero-endpoint process:
    # a stochastic positive path multiplied by remaining time, ending at zero.
    # For cash-default paths the existing independent default time still applies.
    positive_targets = np.where(zero_targets, endpoints[:, 0], endpoints[:, 1])
    if not np.isfinite(endpoints[:, 0]).all() or (endpoints[:, 0] <= 0).any():
        raise ValueError("Starting prices must be finite and positive.")
    sigma = float(daily_volatility)
    if not np.isfinite(sigma) or sigma < 0:
        raise ValueError("Daily volatility must be finite and nonnegative.")

    shock_days = market_shocks["day_number"].to_numpy(dtype=float)
    shock_returns = market_shocks["predicted_stock_return"].to_numpy(dtype=float)
    if (
        not np.isfinite(shock_days).all()
        or (shock_days != np.floor(shock_days)).any()
        or ((shock_days < 1) | (shock_days > n_days)).any()
        or not np.isfinite(shock_returns).all()
        or (shock_returns <= -1).any()
    ):
        raise ValueError("Market shocks require valid day numbers and returns greater than -100%.")
    # Summing log jumps compounds all simple-return shocks on the same day.
    log_jumps = pd.Series(
        np.log1p(shock_returns), index=shock_days.astype(int), dtype=float
    ).groupby(level=0).sum()

    if not isinstance(innovations, pd.DataFrame):
        raise ValueError("Supply the labelled company slice of the shared annual dataset.")
    if not innovations.index.equals(paths.index) or not innovations.columns.equals(dates):
        raise ValueError("Innovation simulation IDs and trading dates must match the paths exactly.")
    daily_innovations = innovations.to_numpy(dtype=float)
    if daily_innovations.shape != (len(paths), n_days) or not np.isfinite(daily_innovations).all():
        raise ValueError("The shared company innovations have invalid dimensions or values.")
    log_prices = np.log(endpoints[:, 0])
    log_targets = np.log(positive_targets)
    # Compensate all known jumps once. This preserves the configured expected
    # terminal price and full-horizon diffusion variance, even on a final-day jump.
    total_log_jump = float(log_jumps.sum())
    log_drift = (log_targets - log_prices - total_log_jump) / n_days - 0.5 * sigma**2
    daily_prices = np.empty((len(paths), n_days), dtype=float)

    for day_number in range(1, n_days + 1):
        log_prices += log_drift + sigma * daily_innovations[:, day_number - 1]
        if day_number in log_jumps.index:
            log_prices += float(log_jumps.loc[day_number])
        daily_prices[:, day_number - 1] = np.exp(log_prices)

    if not np.isfinite(daily_prices).all() or (daily_prices <= 0).any():
        raise ValueError("The simulation produced nonfinite or nonpositive prices.")
    if zero_targets.any():
        daily_prices[zero_targets] *= 1 - np.arange(1, n_days + 1) / n_days
    # Use a separate random stream so ordinary GBM prices stay unchanged.
    default_rng = np.random.default_rng(default_seed)
    default_days = default_rng.integers(1, n_days + 1, size=len(default_rows))
    for row, default_day in zip(default_rows, default_days):
        # Zero is permanent: later market shocks cannot revive a defaulted path.
        daily_prices[row, default_day - 1:] = 0.0

    return pd.DataFrame(
        daily_prices, index=paths.index,
        columns=[f"day_{day}" for day in range(1, n_days + 1)],
    )


daily_prices_df = generate_daily_stock_prices(
    paths_df, trading_dates, market_shocks_df, daily_log_volatility,
    innovations=company_price_innovations, default_seed=2027
)
old_daily_columns = [
    column for column in paths_df.columns
    if isinstance(column, str) and column.startswith("day_") and column[4:].isdigit()
]
paths_df = pd.concat([paths_df.drop(columns=old_daily_columns), daily_prices_df], axis=1)
daily_price_dates = pd.Series(trading_dates, index=daily_prices_df.columns, name="date")
# Bind provenance to the innovations actually used, so stale export cells fail.
from copy import deepcopy
paths_df.attrs["joint_price_innovations"] = deepcopy(
    company_price_innovations.attrs["joint_price_innovations"]
)

print(
    f"Generated {len(paths_df):,} paths with {TRADING_DAYS} daily prices "
    f"({trading_dates[0]:%d.%m.%Y} to {trading_dates[-1]:%d.%m.%Y})."
)
print(f"Paths set to zero from a random trading day: {paths_df['is_default'].sum():,}")
paths_df


### Visualize the stock-price paths

Faint blue lines show paths that survive, and red lines show paths flagged as defaulting because their losses exhaust the cash reserves. The dark line shows the average across all paths. The dotted horizontal line marks the starting share price. Dashed vertical lines mark the trading dates when market shocks are applied.

All paths begin at their starting share price, followed by the simulated daily prices. Defaulting paths stay at zero after their assigned default day. Trading days are equally spaced to avoid weekend and holiday gaps, while the axis labels still show the dates.


In [ ]:
from matplotlib.lines import Line2D

path_chart_columns = daily_price_dates.index.tolist()
path_chart_dates = pd.DatetimeIndex(
    [SIMULATION_START_DATE, *daily_price_dates.loc[path_chart_columns].tolist()]
)
path_chart_positions = np.arange(len(path_chart_dates))
path_chart_prices = np.column_stack([
    paths_df["share_price_start"].to_numpy(dtype=float),
    paths_df[path_chart_columns].to_numpy(dtype=float),
])

path_chart_defaults = paths_df["is_default"].to_numpy(dtype=bool)
path_chart_start_price = float(paths_df["share_price_start"].iloc[0])

fig, ax = plt.subplots(figsize=(12, 6.5), dpi=140)
ax.plot(
    path_chart_positions, path_chart_prices[~path_chart_defaults].T,
    color="#5489bc", alpha=0.025, linewidth=0.45, rasterized=True,
)
if path_chart_defaults.any():
    ax.plot(
        path_chart_positions, path_chart_prices[path_chart_defaults].T,
        color="#d62728", alpha=0.8, linewidth=1.0,
        zorder=3, clip_on=False, rasterized=True,
    )
starting_price_line = ax.axhline(
    path_chart_start_price, color="#4b5563", linestyle=":", linewidth=1.8,
    label=f"Starting price: ${path_chart_start_price:.2f}", zorder=4,
)
mean_price_line, = ax.plot(
    path_chart_positions, path_chart_prices.mean(axis=0),
    color="#122d4a", linewidth=2.5, label="Average price", zorder=4,
)

for shock in market_shocks_df.itertuples(index=False):
    shock_label = "Oil shock" if shock.shock_type == "oil_price_shock" else "Interest-rate shock"
    shock_color = "#b56c17" if shock.shock_type == "oil_price_shock" else "#78519b"
    ax.axvline(shock.day_number, color=shock_color, linestyle="--", linewidth=1.2, zorder=3)
    ax.text(
        shock.day_number, 0.97,
        f"{shock_label}\n{shock.trading_date:%d %b %Y}",
        transform=ax.get_xaxis_transform(), ha="center", va="top",
        color=shock_color, fontsize=9,
        bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.85, "pad": 2},
        zorder=5,
    )

ax.set_title(f"{TICKER} simulated stock-price paths")
ax.set_xlabel("Date (equally spaced trading days)")
ax.set_ylabel("Share price (USD)")
ax.set_xlim(path_chart_positions[0], path_chart_positions[-1])
ax.set_ylim(bottom=0)
path_chart_tick_positions = np.flatnonzero(
    ~path_chart_dates.to_period("M").duplicated()
)[::2]
ax.set_xticks(path_chart_tick_positions)
ax.set_xticklabels(path_chart_dates[path_chart_tick_positions].strftime("%b %Y"))
ax.grid(alpha=0.2)
ax.set_axisbelow(True)
ax.legend(handles=[
    Line2D([0], [0], color="#5489bc", linewidth=1,
           label=f"Surviving paths ({(~path_chart_defaults).sum():,})"),
    Line2D([0], [0], color="#d62728", linewidth=1.5,
           label=f"Defaulting paths ({path_chart_defaults.sum():,})"),
    mean_price_line,
    starting_price_line,
], loc="upper right", frameon=False)
fig.tight_layout()
plt.show()


### Histogram of total path returns

We calculate one total percentage return for each path: `final_simulated_price / share_price_start - 1`. The final price comes from the last `day_N` column. A path ending at zero has a return of **−100%**.

The x-axis shows total returns over the simulation, and the y-axis counts paths. The reference lines mark zero return and the average return. Daily log returns are used for the price simulation; this chart shows the overall percentage gain or loss.


In [ ]:
from matplotlib.ticker import PercentFormatter

final_daily_column = daily_price_dates.index[-1]
path_start_prices = paths_df["share_price_start"].astype(float)
path_final_prices = paths_df[final_daily_column].astype(float)
if (
    paths_df.empty
    or not np.isfinite(path_start_prices).all()
    or not np.isfinite(path_final_prices).all()
    or (path_start_prices <= 0).any()
    or (path_final_prices < 0).any()
):
    raise ValueError("Path returns require positive start prices and nonnegative finite final prices.")

path_returns = path_final_prices / path_start_prices - 1
average_path_return = float(path_returns.mean())

fig, ax = plt.subplots(figsize=(12, 6), dpi=140)
ax.hist(
    path_returns, bins=50, color="#477aa4",
    edgecolor="white", linewidth=0.6,
)
ax.axvline(0, color="#555555", linestyle=":", linewidth=1.8, label="0% return")
ax.axvline(
    average_path_return, color="#122d4a", linestyle="--", linewidth=2,
    label=f"Average return: {average_path_return:+.2%}",
)
ax.set_title(f"{TICKER}: total returns across {len(path_returns):,} simulated paths")
ax.set_xlabel("Total return over the simulation")
ax.set_ylabel("Number of paths")
ax.xaxis.set_major_formatter(PercentFormatter(xmax=1.0))
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.18)
ax.set_axisbelow(True)
ax.legend(frameon=False)
fig.tight_layout()
plt.show()


### Historical and generated daily returns

We compare daily log returns, `ln(price_today / price_previous)`, using the same five years of historical data as the volatility calculation. Generated returns include every path and every trading day, starting with `day_0` to `day_1`, including the immediate market shocks.

Both histograms use the same bins and are scaled to a total area of one. This makes their shapes comparable even though there are many more generated observations. The x-axis shows daily log returns, and the y-axis shows probability density.

A default jump to zero has no finite log return. Such jumps and the zero-price days afterward are counted separately and excluded from this comparison. Ordinary generated returns still include all movements before default.


In [ ]:
from matplotlib.ticker import PercentFormatter

historical_daily_returns = historical_stock_volatility_df["log_return"].to_numpy(dtype=float)
daily_comparison_prices = np.column_stack([
    paths_df["share_price_start"].to_numpy(dtype=float),
    paths_df[daily_price_dates.index.tolist()].to_numpy(dtype=float),
])
if (
    historical_daily_returns.size == 0
    or not np.isfinite(historical_daily_returns).all()
    or daily_comparison_prices.shape[0] == 0
    or not np.isfinite(daily_comparison_prices).all()
    or (daily_comparison_prices < 0).any()
    or (daily_comparison_prices[:, 0] <= 0).any()
):
    raise ValueError("Daily-return comparison requires valid historical returns and nonnegative simulated prices.")

daily_previous_prices = daily_comparison_prices[:, :-1]
daily_next_prices = daily_comparison_prices[:, 1:]
if ((daily_previous_prices == 0) & (daily_next_prices > 0)).any():
    raise ValueError("A defaulted path must remain at zero.")
positive_price_pairs = (daily_previous_prices > 0) & (daily_next_prices > 0)
generated_daily_returns = (
    np.log(daily_next_prices[positive_price_pairs])
    - np.log(daily_previous_prices[positive_price_pairs])
)
generated_default_jumps = int(((daily_previous_prices > 0) & (daily_next_prices == 0)).sum())
generated_post_default_days = int(((daily_previous_prices == 0) & (daily_next_prices == 0)).sum())
if generated_daily_returns.size == 0:
    raise ValueError("No positive-price daily returns are available for comparison.")

daily_comparison_bins = np.histogram_bin_edges(
    np.concatenate([historical_daily_returns, generated_daily_returns]), bins=80,
)
fig, ax = plt.subplots(figsize=(12, 6), dpi=140)
ax.hist(
    historical_daily_returns, bins=daily_comparison_bins, density=True,
    color="#d28a25", alpha=0.35, edgecolor="#b87516", linewidth=0.5,
    label=f"Historical ({len(historical_daily_returns):,} daily returns)",
)
ax.hist(
    generated_daily_returns, bins=daily_comparison_bins, density=True,
    histtype="step", color="#224e80", linewidth=2,
    label=f"Generated ({len(generated_daily_returns):,} daily returns)",
)
ax.axvline(0, color="#64748b", linestyle=":", linewidth=1)
ax.set_title(f"{TICKER}: historical and generated daily log returns")
ax.set_xlabel("Daily log return")
ax.set_ylabel("Probability density")
ax.xaxis.set_major_formatter(PercentFormatter(xmax=1.0))
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.18)
ax.set_axisbelow(True)
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

print(
    f"Excluded from the log-return comparison: {generated_default_jumps:,} zero-price transitions (cash default or zero equity value) "
    f"and {generated_post_default_days:,} subsequent zero-price days."
)


## Export generated paths and income statements

Two verified tables are written under `EXPORT_ROOT/[ticker]/structured/`: `paths.parquet` contains the cash-default flag, starting price, and daily prices; `income_statements.parquet` contains current account values in the same row order. Source account names and null values are preserved.

Both tables are staged and read back before replacing existing exports. A failed write preserves the previous files. Unrelated files and older export folders remain intact. Validation runs use a temporary export directory.

Both Parquet schemas also contain the JSON metadata key `joint_price_innovations`.
It records the shared manifest/array/correlation checksums, fixed company ordering,
simulation ID range, trading dates, seed derivation, historical window and shrinkage.
Use these identifiers before combining companies or scenarios: identical row numbers
alone do not establish that exports came from the same shared annual dataset.
Legacy exports and the unchanged regression exports do not carry this provenance.


In [ ]:
def export_paths_to_parquet(paths, ticker, trading_days, export_root="export", *, innovation_metadata):
    """Export paired price/account tables in row order; return the paths.parquet path."""
    from collections.abc import Mapping
    from numbers import Real
    if not isinstance(trading_days, (int, np.integer)) or isinstance(trading_days, bool) or trading_days < 1:
        raise ValueError("The number of trading days must be a positive integer.")
    if not isinstance(ticker, str) or not ticker.strip() or ticker in {".", ".."} or Path(ticker).name != ticker:
        raise ValueError("The ticker must be a single nonempty folder name.")
    if not paths.columns.is_unique:
        raise ValueError("The dataframe must have unique column names.")

    daily_columns = [f"day_{day}" for day in range(1, trading_days + 1)]
    source_columns = ["is_default", "share_price_start", *daily_columns]
    missing_columns = [column for column in [*source_columns, "income_statement"] if column not in paths.columns]
    if missing_columns:
        raise ValueError(f"Required export columns are missing: {missing_columns}")

    export_data = paths.loc[:, source_columns].copy().rename(
        columns={"share_price_start": "day_0"}
    )
    if (
        not pd.api.types.is_bool_dtype(export_data["is_default"].dtype)
        or export_data["is_default"].isna().any()
    ):
        raise ValueError("Every exported path must have a True or False default flag.")
    price_values = export_data[["day_0", *daily_columns]].to_numpy(dtype=float)
    if (
        export_data.empty or not np.isfinite(price_values).all()
        or (price_values < 0).any() or (price_values[:, 0] <= 0).any()
    ):
        raise ValueError("Export requires positive starting prices and nonnegative finite daily prices.")

    # Use original account names, but evaluate the current model values.
    # This includes the applied shocks and calculated totals, taxes, and profit.
    first_model = paths["income_statement"].iloc[0]
    first_accounts = getattr(first_model, "initial_amounts", None)
    if not isinstance(first_accounts, Mapping) or not first_accounts:
        raise ValueError("Each income statement must provide its original account schema.")
    account_paths = list(first_accounts)
    if any(
        not isinstance(account_path, tuple) or not account_path
        or any(
            not isinstance(part, str) or not part.isidentifier() or part.startswith("_")
            for part in account_path
        )
        for account_path in account_paths
    ):
        raise ValueError("Income-statement account paths must contain valid account names.")
    account_columns = [".".join(account_path) for account_path in account_paths]
    account_schema = set(account_paths)
    income_statement_rows = []

    for row_number, model in enumerate(paths["income_statement"]):
        model_accounts = getattr(model, "initial_amounts", None)
        if not isinstance(model_accounts, Mapping) or set(model_accounts) != account_schema:
            raise ValueError(f"Income statement at row {row_number} has a different account schema.")
        row = {}
        for account_path, column in zip(account_paths, account_columns):
            account = model.IncomeStatement
            for part in account_path:
                account = getattr(account, part)
            value = account()
            if value is not None and (
                isinstance(value, bool) or not isinstance(value, Real) or not np.isfinite(value)
            ):
                raise ValueError(f"Income statement at row {row_number}: {column} must be numeric or None.")
            row[column] = value
        income_statement_rows.append(row)

    income_statements_data = pd.DataFrame(income_statement_rows, columns=account_columns)

    if (
        paths.attrs.get("joint_price_innovations") != innovation_metadata
        or innovation_metadata.get("company") != ticker
        or innovation_metadata["config"]["n_simulations"] != len(paths)
        or len(innovation_metadata["trading_dates"]) != trading_days
        or not paths.index.equals(pd.RangeIndex(len(paths)))
    ):
        raise ValueError("Export rows, dates and company must match shared-innovation provenance.")

    export_directory = Path(export_root).expanduser().resolve() / ticker / "structured"
    export_directory.mkdir(parents=True, exist_ok=True)
    export_file = export_directory / "paths.parquet"
    income_statements_file = export_directory / "income_statements.parquet"

    notebook_support["write_parquet_tables"]({
        "paths.parquet": export_data,
        "income_statements.parquet": income_statements_data,
    }, export_directory, metadata={"joint_price_innovations": innovation_metadata})
    return export_file


exported_paths_file = export_paths_to_parquet(
    paths_df, TICKER, TRADING_DAYS, export_root=EXPORT_ROOT,
    innovation_metadata=joint_innovation_metadata,
)
print(f"Exported {len(paths_df):,} paths to: {exported_paths_file}")
exported_income_statements_file = exported_paths_file.with_name("income_statements.parquet")
print(f"Exported {len(paths_df):,} income statements to: {exported_income_statements_file}")
